In [3]:

# RECOVERY ANALYSIS


from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)

Project root: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics
Processed data: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed


In [4]:
# LOAD GOLDEN ACCOUNTS AND PAYMENTS

accounts = pd.read_csv(
    PROCESSED_DIR / "golden_accounts.csv",
    low_memory=False
)

payments = pd.read_csv(
    PROCESSED_DIR / "golden_payments.csv",
    low_memory=False
)

print("Golden accounts:", len(accounts))
print("Golden payments:", len(payments))


Golden accounts: 30000
Golden payments: 24528


In [5]:
# RECOVERED AMOUNT

successful_payments = payments[
    payments["payment_status"].eq("SUCCESS")
].copy()

recovered_amount = successful_payments["amount"].sum()

successful_payment_count = (
    successful_payments["payment_id"].nunique()
)

print(
    f"Successful payment records: {successful_payment_count:,}"
)

print(
    f"Recovered amount: ₹{recovered_amount:,.2f}"
)

Successful payment records: 17,199
Recovered amount: ₹1,291,463,001.95


In [6]:
# OUTSTANDING PORTFOLIO
total_outstanding = accounts[
    "outstanding_amount"
].sum()

account_count = accounts[
    "account_id"
].nunique()

print(
    f"Accounts: {account_count:,}"
)

print(
    f"Total outstanding amount: ₹{total_outstanding:,.2f}"
)

Accounts: 30,000
Total outstanding amount: ₹10,489,035,343.00


In [7]:
# CORE RECOVERY RATE
recovery_rate = (
    recovered_amount
    / total_outstanding
    * 100
)

print(
    f"Recovery rate: {recovery_rate:.2f}%"
)

Recovery rate: 12.31%


In [8]:
# METRIC DEFINITIONS
metric_definitions = pd.DataFrame([
    {
        "metric": "Recovered Amount",
        "numerator": "Sum of amount where payment_status = SUCCESS",
        "denominator": "N/A",
        "grain": "Payment"
    },
    {
        "metric": "Recovery Rate",
        "numerator": "Successful payment amount",
        "denominator": "Total outstanding amount in accounts",
        "grain": "Portfolio"
    }
])

metric_definitions

,metric,numerator,denominator,grain
0,Recovered Amount,Sum of amount where payment_status = SUCCESS,N/A,Payment
1,Recovery Rate,Successful payment amount,Total outstanding amount in accounts,Portfolio


In [9]:
# SAVE CORE RECOVERY METRICS
recovery_summary = pd.DataFrame([{
    "accounts": account_count,
    "total_outstanding": total_outstanding,
    "successful_payment_count": successful_payment_count,
    "recovered_amount": recovered_amount,
    "recovery_rate_pct": recovery_rate
}])

output_file = (
    PROCESSED_DIR / "recovery_summary.csv"
)

recovery_summary.to_csv(
    output_file,
    index=False
)

recovery_summary


,accounts,total_outstanding,successful_payment_count,recovered_amount,recovery_rate_pct
0,30000,1.048904e+10,17199,1.291463e+09,12.312505


In [10]:
#  PREPARE PAYMENT DATES
payments["event_at_dt"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

payments["payment_month"] = (
    payments["event_at_dt"]
    .dt.to_period("M")
)

print(
    "Valid payment timestamps:",
    payments["event_at_dt"].notna().sum()
)

print(
    "Invalid payment timestamps:",
    payments["event_at_dt"].isna().sum()
)


Valid payment timestamps: 24528
Invalid payment timestamps: 0


In [11]:
#  MONTHLY SUCCESSFUL RECOVERY
monthly_recovery = (
    payments[
        payments["payment_status"].eq("SUCCESS")
        & payments["payment_month"].notna()
    ]
    .groupby("payment_month")
    .agg(
        successful_payment_count=("payment_id", "nunique"),
        recovered_amount=("amount", "sum")
    )
    .reset_index()
)

monthly_recovery


,payment_month,successful_payment_count,recovered_amount
0,2026-01,2413,1.833250e+08
1,2026-02,2223,1.664619e+08
2,2026-03,2475,1.851473e+08
3,2026-04,2358,1.720308e+08
4,2026-05,2413,1.816228e+08
5,2026-06,2325,1.729824e+08
6,2026-07,2397,1.842168e+08
7,2026-08,595,4.567592e+07


In [12]:
#  MONTHLY PAYMENT STATUS MIX
monthly_status = (
    payments[
        payments["payment_month"].notna()
    ]
    .groupby(
        ["payment_month", "payment_status"]
    )
    .agg(
        payment_records=("payment_id", "count"),
        payment_amount=("amount", "sum")
    )
    .reset_index()
)

monthly_status

,payment_month,payment_status,payment_records,payment_amount
0,2026-01,FAILED,502,3.700555e+07
1,2026-01,PENDING,366,2.792243e+07
2,2026-01,REVERSED,172,1.146519e+07
3,2026-01,SUCCESS,2413,1.833250e+08
4,2026-02,FAILED,443,3.433572e+07
5,2026-02,PENDING,302,2.211157e+07
6,2026-02,REVERSED,138,1.119887e+07
7,2026-02,SUCCESS,2225,1.664619e+08
8,2026-03,FAILED,528,4.049808e+07
9,2026-03,PENDING,324,2.405330e+07


In [13]:
#  CHECK FOR HISTORICAL BALANCE DATA
print("Account columns:")
print(accounts.columns.tolist())

print("\nPotential balance/outstanding fields:")

balance_columns = [
    col for col in accounts.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "outstanding",
            "balance",
            "principal",
            "amount"
        ]
    )
]

print(balance_columns)

Account columns:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version', 'borrower_id_check', 'quality_flag', 'latest_historical_status', 'status_history_match']

Potential balance/outstanding fields:
['principal_amount', 'outstanding_amount']


In [14]:
#  MONTHLY RECOVERY SUMMARY
monthly_recovery["recovered_amount"] = (
    monthly_recovery["recovered_amount"].round(2)
)

monthly_recovery["month"] = (
    monthly_recovery["payment_month"]
    .astype(str)
)

monthly_recovery = monthly_recovery[
    [
        "month",
        "successful_payment_count",
        "recovered_amount"
    ]
].sort_values("month")

display(monthly_recovery)


,month,successful_payment_count,recovered_amount
0,2026-01,2413,1.833250e+08
1,2026-02,2223,1.664619e+08
2,2026-03,2475,1.851473e+08
3,2026-04,2358,1.720308e+08
4,2026-05,2413,1.816228e+08
5,2026-06,2325,1.729824e+08
6,2026-07,2397,1.842168e+08
7,2026-08,595,4.567592e+07


In [15]:
#  MONTH-ON-MONTH RECOVERY MOVEMENT
monthly_recovery["recovery_change"] = (
    monthly_recovery["recovered_amount"].diff()
)

monthly_recovery["recovery_change_pct"] = (
    monthly_recovery["recovered_amount"]
    .pct_change()
    * 100
)

monthly_recovery

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct
0,2026-01,2413,1.833250e+08,NaN,NaN
1,2026-02,2223,1.664619e+08,-1.686303e+07,-9.198435
2,2026-03,2475,1.851473e+08,1.868533e+07,11.224984
3,2026-04,2358,1.720308e+08,-1.311643e+07,-7.084320
4,2026-05,2413,1.816228e+08,9.591980e+06,5.575733
5,2026-06,2325,1.729824e+08,-8.640417e+06,-4.757341
6,2026-07,2397,1.842168e+08,1.123442e+07,6.494544
7,2026-08,595,4.567592e+07,-1.385409e+08,-75.205347


In [16]:
#  MONTHLY PAYMENT COUNT MOVEMENT
monthly_recovery["payment_count_change"] = (
    monthly_recovery["successful_payment_count"].diff()
)

monthly_recovery["payment_count_change_pct"] = (
    monthly_recovery["successful_payment_count"]
    .pct_change()
    * 100
)

monthly_recovery

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct,payment_count_change,payment_count_change_pct
0,2026-01,2413,1.833250e+08,NaN,NaN,NaN,NaN
1,2026-02,2223,1.664619e+08,-1.686303e+07,-9.198435,-190.0,-7.874016
2,2026-03,2475,1.851473e+08,1.868533e+07,11.224984,252.0,11.336032
3,2026-04,2358,1.720308e+08,-1.311643e+07,-7.084320,-117.0,-4.727273
4,2026-05,2413,1.816228e+08,9.591980e+06,5.575733,55.0,2.332485
5,2026-06,2325,1.729824e+08,-8.640417e+06,-4.757341,-88.0,-3.646913
6,2026-07,2397,1.842168e+08,1.123442e+07,6.494544,72.0,3.096774
7,2026-08,595,4.567592e+07,-1.385409e+08,-75.205347,-1802.0,-75.177305


In [17]:
#  MONTHLY AVERAGE SUCCESSFUL PAYMENT
monthly_recovery["avg_successful_payment"] = (
    monthly_recovery["recovered_amount"]
    / monthly_recovery["successful_payment_count"]
)

monthly_recovery

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct,payment_count_change,payment_count_change_pct,avg_successful_payment
0,2026-01,2413,1.833250e+08,NaN,NaN,NaN,NaN,75973.879304
1,2026-02,2223,1.664619e+08,-1.686303e+07,-9.198435,-190.0,-7.874016,74881.665308
2,2026-03,2475,1.851473e+08,1.868533e+07,11.224984,252.0,11.336032,74806.977321
3,2026-04,2358,1.720308e+08,-1.311643e+07,-7.084320,-117.0,-4.727273,72956.252455
4,2026-05,2413,1.816228e+08,9.591980e+06,5.575733,55.0,2.332485,75268.472018
5,2026-06,2325,1.729824e+08,-8.640417e+06,-4.757341,-88.0,-3.646913,74401.034951
6,2026-07,2397,1.842168e+08,1.123442e+07,6.494544,72.0,3.096774,76853.076813
7,2026-08,595,4.567592e+07,-1.385409e+08,-75.205347,-1802.0,-75.177305,76766.256622


In [18]:
#  SAVE MONTHLY RECOVERY ANALYSIS
monthly_output = (
    PROCESSED_DIR / "monthly_recovery.csv"
)

monthly_recovery.to_csv(
    monthly_output,
    index=False
)

print("Saved:", monthly_output)

Saved: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed\monthly_recovery.csv


In [19]:
#  CORE KPI SUMMARY



core_kpis = pd.DataFrame([
    {
        "metric": "Total Accounts",
        "value": accounts["account_id"].nunique()
    },
    {
        "metric": "Total Outstanding Amount",
        "value": accounts["outstanding_amount"].sum()
    },
    {
        "metric": "Successful Payment Records",
        "value": payments[
            payments["payment_status"].eq("SUCCESS")
        ]["payment_id"].nunique()
    },
    {
        "metric": "Recovered Amount",
        "value": payments[
            payments["payment_status"].eq("SUCCESS")
        ]["amount"].sum()
    },
    {
        "metric": "Recovery Rate",
        "value": recovery_rate
    }
])

core_kpis

,metric,value
0,Total Accounts,3.000000e+04
1,Total Outstanding Amount,1.048904e+10
2,Successful Payment Records,1.719900e+04
3,Recovered Amount,1.291463e+09
4,Recovery Rate,1.231251e+01


In [20]:
core_kpis.to_csv(
    PROCESSED_DIR / "core_kpis.csv",
    index=False
)

In [21]:
#  DPD BUCKETS
def dpd_bucket(dpd):
    if pd.isna(dpd):
        return "Unknown"
    elif dpd <= 30:
        return "0-30"
    elif dpd <= 60:
        return "31-60"
    elif dpd <= 90:
        return "61-90"
    elif dpd <= 120:
        return "91-120"
    else:
        return "120+"

accounts["dpd_bucket"] = accounts["dpd"].apply(dpd_bucket)

display(
    accounts[
        ["account_id", "dpd", "dpd_bucket"]
    ].head(20)
)


,account_id,dpd,dpd_bucket
0,ACC0000001,15,0-30
1,ACC0000002,5,0-30
2,ACC0000003,60,31-60
3,ACC0000004,30,0-30
4,ACC0000005,180,120+
5,ACC0000006,45,31-60
6,ACC0000007,45,31-60
7,ACC0000008,1,0-30
8,ACC0000009,0,0-30
9,ACC0000010,120,91-120


In [22]:
#  PAYMENT → ACCOUNT → DPD
payment_dpd = payments.merge(
    accounts[
        [
            "account_id",
            "dpd",
            "dpd_bucket",
            "risk_segment",
            "loan_type",
            "outstanding_amount"
        ]
    ],
    on="account_id",
    how="left",
    validate="many_to_one"
)

print("Payment rows:", len(payments))
print("After account join:", len(payment_dpd))

assert len(payment_dpd) == len(payments)

Payment rows: 24528
After account join: 24528


In [23]:
# RECOVERY BY DPD BUCKET
recovery_by_dpd = (
    payment_dpd[
        payment_dpd["payment_status"].eq("SUCCESS")
    ]
    .groupby("dpd_bucket", dropna=False)
    .agg(
        successful_payments=("payment_id", "nunique"),
        recovered_amount=("amount", "sum"),
        accounts_with_payment=("account_id", "nunique")
    )
    .reset_index()
)

recovery_by_dpd = recovery_by_dpd.sort_values(
    "recovered_amount",
    ascending=False
)

recovery_by_dpd

,dpd_bucket,successful_payments,recovered_amount,accounts_with_payment
0,0-30,7744,5.785240e+08,5855
2,31-60,3281,2.476348e+08,2488
3,61-90,3108,2.344337e+08,2413
4,91-120,1574,1.212158e+08,1207
1,120+,1492,1.096546e+08,1146


In [24]:
#  PORTFOLIO MIX BY DPD
portfolio_by_dpd = (
    accounts
    .groupby("dpd_bucket", dropna=False)
    .agg(
        accounts=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

portfolio_by_dpd

,dpd_bucket,accounts,outstanding_amount
0,0-30,13565,4.782319e+09
1,120+,2694,9.204078e+08
2,31-60,5514,1.914490e+09
3,61-90,5468,1.895868e+09
4,91-120,2759,9.759497e+08


In [25]:
#  DPD RECOVERY VIEW



dpd_recovery_view = portfolio_by_dpd.merge(
    recovery_by_dpd,
    on="dpd_bucket",
    how="left"
)

dpd_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
] = dpd_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
].fillna(0)

dpd_recovery_view["recovered_amount_share_pct"] = (
    dpd_recovery_view["recovered_amount"]
    / dpd_recovery_view["recovered_amount"].sum()
    * 100
)

dpd_recovery_view["portfolio_share_pct"] = (
    dpd_recovery_view["outstanding_amount"]
    / dpd_recovery_view["outstanding_amount"].sum()
    * 100
)

dpd_recovery_view


,dpd_bucket,accounts,outstanding_amount,successful_payments,recovered_amount,accounts_with_payment,recovered_amount_share_pct,portfolio_share_pct
0,0-30,13565,4.782319e+09,7744,5.785240e+08,5855,44.796020,45.593508
1,120+,2694,9.204078e+08,1492,1.096546e+08,1146,8.490729,8.774951
2,31-60,5514,1.914490e+09,3281,2.476348e+08,2488,19.174750,18.252301
3,61-90,5468,1.895868e+09,3108,2.344337e+08,2413,18.152571,18.074764
4,91-120,2759,9.759497e+08,1574,1.212158e+08,1207,9.385930,9.304476


In [26]:
#  ACCOUNT-LEVEL PAYMENT CONVERSION BY DPD
dpd_recovery_view["account_payment_conversion_pct"] = (
    dpd_recovery_view["accounts_with_payment"]
    / dpd_recovery_view["accounts"]
    * 100
)

dpd_recovery_view[
    [
        "dpd_bucket",
        "accounts",
        "outstanding_amount",
        "accounts_with_payment",
        "account_payment_conversion_pct",
        "recovered_amount"
    ]
]

,dpd_bucket,accounts,outstanding_amount,accounts_with_payment,account_payment_conversion_pct,recovered_amount
0,0-30,13565,4.782319e+09,5855,43.162551,5.785240e+08
1,120+,2694,9.204078e+08,1146,42.538976,1.096546e+08
2,31-60,5514,1.914490e+09,2488,45.121509,2.476348e+08
3,61-90,5468,1.895868e+09,2413,44.129481,2.344337e+08
4,91-120,2759,9.759497e+08,1207,43.747735,1.212158e+08


In [27]:
# PORTFOLIO MIX BY RISK SEGMENT
portfolio_by_risk = (
    accounts
    .groupby("risk_segment", dropna=False)
    .agg(
        accounts=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

portfolio_by_risk["portfolio_share_pct"] = (
    portfolio_by_risk["outstanding_amount"]
    / portfolio_by_risk["outstanding_amount"].sum()
    * 100
)

portfolio_by_risk

,risk_segment,accounts,outstanding_amount,portfolio_share_pct
0,HIGH,7552,2.646183e+09,25.228083
1,LOW,7513,2.633311e+09,25.105370
2,MEDIUM,7533,2.628179e+09,25.056446
3,NPA,7402,2.581362e+09,24.610101


In [28]:
#  RECOVERY BY RISK SEGMENT
recovery_by_risk = (
    payment_dpd[
        payment_dpd["payment_status"].eq("SUCCESS")
    ]
    .groupby("risk_segment", dropna=False)
    .agg(
        successful_payments=("payment_id", "nunique"),
        recovered_amount=("amount", "sum"),
        accounts_with_payment=("account_id", "nunique")
    )
    .reset_index()
)

recovery_by_risk

,risk_segment,successful_payments,recovered_amount,accounts_with_payment
0,HIGH,4284,3.252772e+08,3266
1,LOW,4383,3.255906e+08,3340
2,MEDIUM,4336,3.239476e+08,3283
3,NPA,4196,3.166476e+08,3220


In [29]:

#  RISK SEGMENT RECOVERY VIEW


risk_recovery_view = portfolio_by_risk.merge(
    recovery_by_risk,
    on="risk_segment",
    how="left"
)

risk_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
] = risk_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
].fillna(0)

risk_recovery_view["account_payment_conversion_pct"] = (
    risk_recovery_view["accounts_with_payment"]
    / risk_recovery_view["accounts"]
    * 100
)

risk_recovery_view["recovery_share_pct"] = (
    risk_recovery_view["recovered_amount"]
    / risk_recovery_view["recovered_amount"].sum()
    * 100
)

risk_recovery_view

,risk_segment,accounts,outstanding_amount,portfolio_share_pct,successful_payments,recovered_amount,accounts_with_payment,account_payment_conversion_pct,recovery_share_pct
0,HIGH,7552,2.646183e+09,25.228083,4284,3.252772e+08,3266,43.246822,25.186723
1,LOW,7513,2.633311e+09,25.105370,4383,3.255906e+08,3340,44.456276,25.210989
2,MEDIUM,7533,2.628179e+09,25.056446,4336,3.239476e+08,3283,43.581574,25.083767
3,NPA,7402,2.581362e+09,24.610101,4196,3.166476e+08,3220,43.501756,24.518520


In [30]:

#  RECOVERY BY LOAN TYPE


portfolio_by_loan = (
    accounts
    .groupby("loan_type", dropna=False)
    .agg(
        accounts=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

recovery_by_loan = (
    payment_dpd[
        payment_dpd["payment_status"].eq("SUCCESS")
    ]
    .groupby("loan_type", dropna=False)
    .agg(
        successful_payments=("payment_id", "nunique"),
        recovered_amount=("amount", "sum"),
        accounts_with_payment=("account_id", "nunique")
    )
    .reset_index()
)

loan_recovery_view = portfolio_by_loan.merge(
    recovery_by_loan,
    on="loan_type",
    how="left"
)

loan_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
] = loan_recovery_view[
    [
        "successful_payments",
        "recovered_amount",
        "accounts_with_payment"
    ]
].fillna(0)

loan_recovery_view["account_payment_conversion_pct"] = (
    loan_recovery_view["accounts_with_payment"]
    / loan_recovery_view["accounts"]
    * 100
)

loan_recovery_view

,loan_type,accounts,outstanding_amount,successful_payments,recovered_amount,accounts_with_payment,account_payment_conversion_pct
0,AUTO,6079,2.135301e+09,3490,2.632792e+08,2663,43.806547
1,BNPL,5928,2.058915e+09,3343,2.502714e+08,2547,42.965587
2,CONSUMER,5930,2.073113e+09,3499,2.617468e+08,2648,44.654300
3,CREDIT_CARD,6080,2.126677e+09,3481,2.640503e+08,2656,43.684211
4,PERSONAL,5983,2.095030e+09,3386,2.521153e+08,2595,43.372890


In [31]:
#  RECOVERY BY DPD AND RISK SEGMENT
dpd_risk_view = (
    payment_dpd[
        payment_dpd["payment_status"].eq("SUCCESS")
    ]
    .groupby(
        ["dpd_bucket", "risk_segment"],
        dropna=False
    )
    .agg(
        successful_payments=("payment_id", "nunique"),
        recovered_amount=("amount", "sum"),
        accounts_with_payment=("account_id", "nunique")
    )
    .reset_index()
)

dpd_risk_view

,dpd_bucket,risk_segment,successful_payments,recovered_amount,accounts_with_payment
0,0-30,HIGH,1986,1.505187e+08,1501
1,0-30,LOW,1915,1.418773e+08,1442
2,0-30,MEDIUM,1982,1.455536e+08,1500
3,0-30,NPA,1861,1.405744e+08,1412
4,120+,HIGH,359,2.709985e+07,282
5,120+,LOW,420,3.110013e+07,314
6,120+,MEDIUM,346,2.441614e+07,264
7,120+,NPA,367,2.703851e+07,286
8,31-60,HIGH,777,6.077404e+07,577
9,31-60,LOW,873,6.485815e+07,671


In [32]:
#  SAVE RECOVERY SEGMENT ANALYSIS
dpd_recovery_view.to_csv(
    PROCESSED_DIR / "recovery_by_dpd.csv",
    index=False
)

risk_recovery_view.to_csv(
    PROCESSED_DIR / "recovery_by_risk_segment.csv",
    index=False
)

loan_recovery_view.to_csv(
    PROCESSED_DIR / "recovery_by_loan_type.csv",
    index=False
)

dpd_risk_view.to_csv(
    PROCESSED_DIR / "recovery_by_dpd_risk.csv",
    index=False
)

print("Recovery segment analyses saved.")

Recovery segment analyses saved.


In [33]:

# LOAD RAW DATASETS FOR RECOVERY ANALYSIS


datasets = {}

for file in sorted(RAW_DIR.glob("*.csv")):
    datasets[file.stem] = pd.read_csv(
        file,
        low_memory=False
    )

print(f"Datasets loaded: {len(datasets)}")

for name, df in datasets.items():
    print(
        f"{name}: {len(df):,} rows × {len(df.columns)} columns"
    )

Datasets loaded: 18
account_status_history: 60,000 rows × 8 columns
accounts: 30,000 rows × 11 columns
agent_sessions: 15,000 rows × 7 columns
agents: 30,000 rows × 8 columns
borrowers: 30,600 rows × 8 columns
call_attempts: 120,000 rows × 9 columns
call_dispositions: 35,000 rows × 8 columns
calls: 91,350 rows × 11 columns
campaigns: 120 rows × 7 columns
complaints: 8,000 rows × 9 columns
daily_targeting: 45,000 rows × 7 columns
data_dictionary: 143 rows × 3 columns
field_visits: 25,000 rows × 10 columns
payments: 25,500 rows × 9 columns
promises_to_pay: 18,000 rows × 9 columns
sms_events: 45,000 rows × 8 columns
vendor_telephony: 15 rows × 6 columns
whatsapp_events: 60,600 rows × 8 columns


In [34]:
#  TARGETING POPULATION
targeting = datasets["daily_targeting"].copy()

print("Targeting records:", len(targeting))
print("Unique accounts targeted:", targeting["account_id"].nunique())

display(targeting.head())

Targeting records: 45000
Unique accounts targeted: 23344


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED


In [35]:

#  TARGETING STATUS


targeting_status = (
    targeting["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="records")
)

targeting_status["share_pct"] = (
    targeting_status["records"]
    / targeting_status["records"].sum()
    * 100
)

targeting_status

,status,records,share_pct
0,EXPIRED,11371,25.268889
1,CONTACTED,11254,25.008889
2,QUEUED,11202,24.893333
3,SKIPPED,11173,24.828889


In [36]:

#  COLLECTION ATTEMPTS


attempts = datasets["call_attempts"].copy()

print("Total call attempts:", len(attempts))
print("Accounts with attempts:", attempts["account_id"].nunique())

display(attempts.head())

Total call attempts: 120000
Accounts with attempts: 29451


,attempt_id,account_id,borrower_id,event_at,call_id,agent_id,attempt_no,vendor_id,attempt_status
0,ATTEMPT0000001,ACC0009527,BRW0000705,2026-03-10 21:31:13,CALL0019052,AGT0000685,5,VND0000013,FAILED
1,ATTEMPT0000002,ACC0007960,BRW0002732,2026-05-01 20:55:02,CALL0011950,AGT0000472,4,VND0000006,CONNECTED
2,ATTEMPT0000003,ACC0016476,BRW0002174,2026-06-05 20:57:37,CALL0017411,AGT0000035,1,VND0000015,RINGING
3,ATTEMPT0000004,ACC0011111,BRW0003738,2026-05-02 06:08:39,CALL0068999,AGT0000277,3,VND0000010,FAILED
4,ATTEMPT0000005,ACC0010343,BRW0003352,2026-05-27 03:48:23,CALL0049585,AGT0000346,5,VND0000008,CONNECTED


In [37]:

#  CALL STATUS DISTRIBUTION


calls_analysis = datasets["calls"].copy()

call_status = (
    calls_analysis["call_status"]
    .value_counts(dropna=False)
    .rename_axis("call_status")
    .reset_index(name="call_count")
)

call_status["share_pct"] = (
    call_status["call_count"]
    / call_status["call_count"].sum()
    * 100
)

call_status

,call_status,call_count,share_pct
0,NO_ANSWER,18363,20.101806
1,BUSY,18330,20.065681
2,FAILED,18276,20.006568
3,VOICEMAIL,18235,19.961686
4,ANSWERED,18146,19.864258


In [38]:

#  PROMISE-TO-PAY POPULATION


ptp = datasets["promises_to_pay"].copy()

print("PTP records:", len(ptp))
print("Accounts with PTP:", ptp["account_id"].nunique())

print("\nPTP columns:")
print(ptp.columns.tolist())

display(ptp.head(10))

PTP records: 18000
Accounts with PTP: 13532

PTP columns:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS
5,PTP0000006,ACC0020223,BRW0004429,2026-06-14 00:43:18,AGT0000243,51103.49,2026-06-19 00:43:18,BROKEN,WHATSAPP
6,PTP0000007,ACC0018006,BRW0008673,2026-06-04 12:43:41,AGT0000973,18575.15,2026-06-24 12:43:41,CANCELLED,CALL
7,PTP0000008,ACC0011737,BRW0007288,2026-05-17 14:25:29,AGT0000044,64326.80,2026-05-31 14:25:29,OPEN,CALL
8,PTP0000009,ACC0021229,BRW0007682,2026-08-02 10:41:38,AGT0000149,56608.95,2026-08-15 10:41:38,BROKEN,WHATSAPP
9,PTP0000010,ACC0028901,BRW0009589,2026-02-05 15:34:26,AGT0000110,55403.24,2026-02-20 15:34:26,KEPT,SMS


In [39]:

#  PTP STATUS DISTRIBUTION


ptp_status_candidates = [
    col for col in ptp.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "status",
            "outcome",
            "result"
        ]
    )
]

print(
    "Potential PTP status/outcome fields:",
    ptp_status_candidates
)

for col in ptp_status_candidates:
    print(f"\n{col}")
    display(
        ptp[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="records")
    )

Potential PTP status/outcome fields: ['status']

status


,status,records
0,BROKEN,4553
1,CANCELLED,4543
2,KEPT,4489
3,OPEN,4415


In [40]:

#  ATTEMPT → SUCCESSFUL PAYMENT CONVERSION


attempt_accounts = set(
    attempts["account_id"]
    .dropna()
    .astype(str)
)

payment_accounts = set(
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .astype(str)
)

attempt_accounts_with_payment = (
    attempt_accounts & payment_accounts
)

payment_conversion_summary = pd.DataFrame([
    {
        "metric": "Accounts with collection attempts",
        "count": len(attempt_accounts)
    },
    {
        "metric": "Attempted accounts with successful payment",
        "count": len(attempt_accounts_with_payment)
    },
    {
        "metric": "Payment conversion rate",
        "count": (
            len(attempt_accounts_with_payment)
            / len(attempt_accounts)
            * 100
            if len(attempt_accounts)
            else 0
        )
    }
])

payment_conversion_summary

,metric,count
0,Accounts with collection attempts,29451.000000
1,Attempted accounts with successful payment,12872.000000
2,Payment conversion rate,43.706496


In [41]:

#  ATTEMPT → SUCCESSFUL PAYMENT CONVERSION


attempt_accounts = set(
    attempts["account_id"]
    .dropna()
    .astype(str)
)

payment_accounts = set(
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .astype(str)
)

attempt_accounts_with_payment = (
    attempt_accounts & payment_accounts
)

payment_conversion_summary = pd.DataFrame([
    {
        "metric": "Accounts with collection attempts",
        "count": len(attempt_accounts)
    },
    {
        "metric": "Attempted accounts with successful payment",
        "count": len(attempt_accounts_with_payment)
    },
    {
        "metric": "Payment conversion rate",
        "count": (
            len(attempt_accounts_with_payment)
            / len(attempt_accounts)
            * 100
            if len(attempt_accounts)
            else 0
        )
    }
])

payment_conversion_summary

,metric,count
0,Accounts with collection attempts,29451.000000
1,Attempted accounts with successful payment,12872.000000
2,Payment conversion rate,43.706496


In [42]:

#  CALL STATUS VALUES


print("CALL STATUS VALUES")
print("=" * 70)

display(
    calls_analysis["call_status"]
    .value_counts(dropna=False)
    .rename_axis("call_status")
    .reset_index(name="call_count")
)

CALL STATUS VALUES


,call_status,call_count
0,NO_ANSWER,18363
1,BUSY,18330
2,FAILED,18276
3,VOICEMAIL,18235
4,ANSWERED,18146


In [43]:

#  PTP OUTCOME VALUES
print("PTP COLUMNS:")
print(ptp.columns.tolist())

print("\nPTP VALUE DISTRIBUTIONS:")

for col in ptp.columns:
    if ptp[col].dtype == "object":
        print(f"\n--- {col} ---")
        display(
            ptp[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="records")
        )

PTP COLUMNS:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']

PTP VALUE DISTRIBUTIONS:


In [44]:

#  BASIC COLLECTION FUNNEL


targeted_accounts = (
    targeting["account_id"]
    .dropna()
    .astype(str)
    .nunique()
)

attempted_accounts = (
    attempts["account_id"]
    .dropna()
    .astype(str)
    .nunique()
)

successful_payment_accounts = (
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .astype(str)
    .nunique()
)

funnel_basic = pd.DataFrame([
    {
        "stage": "Targeted Accounts",
        "accounts": targeted_accounts
    },
    {
        "stage": "Attempted Accounts",
        "accounts": attempted_accounts
    },
    {
        "stage": "Accounts with Successful Payment",
        "accounts": successful_payment_accounts
    }
])

funnel_basic

,stage,accounts
0,Targeted Accounts,23344
1,Attempted Accounts,29451
2,Accounts with Successful Payment,13109


In [45]:

# 39. BASIC FUNNEL CONVERSION


funnel_basic["conversion_from_previous_stage_pct"] = (
    funnel_basic["accounts"]
    .div(funnel_basic["accounts"].shift(1))
    .mul(100)
)

funnel_basic

,stage,accounts,conversion_from_previous_stage_pct
0,Targeted Accounts,23344,NaN
1,Attempted Accounts,29451,126.160898
2,Accounts with Successful Payment,13109,44.511222


In [46]:

#  CALL STATUS BUSINESS REVIEW TABLE


call_status_review = (
    calls_analysis
    .groupby("call_status", dropna=False)
    .agg(
        calls=("call_id", "nunique"),
        accounts=("account_id", "nunique")
    )
    .reset_index()
    .sort_values("calls", ascending=False)
)

call_status_review

,call_status,calls,accounts
3,NO_ANSWER,18081,13568
1,BUSY,18060,13549
2,FAILED,17999,13505
4,VOICEMAIL,17980,13423
0,ANSWERED,17880,13535


In [47]:

#  PTP ACCOUNT POPULATION


ptp_accounts = (
    ptp["account_id"]
    .dropna()
    .astype(str)
    .nunique()
)

print(
    "Unique accounts with PTP:",
    ptp_accounts
)

Unique accounts with PTP: 13532


In [48]:

#  PTP → SUCCESSFUL PAYMENT


ptp_account_ids = set(
    ptp["account_id"]
    .dropna()
    .astype(str)
)

successful_payment_account_ids = set(
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .astype(str)
)

ptp_with_success = (
    ptp_account_ids
    & successful_payment_account_ids
)

ptp_payment_conversion = (
    len(ptp_with_success)
    / len(ptp_account_ids)
    * 100
    if ptp_account_ids
    else 0
)

print("PTP accounts:", len(ptp_account_ids))
print(
    "PTP accounts with successful payment:",
    len(ptp_with_success)
)
print(
    f"PTP → payment conversion: "
    f"{ptp_payment_conversion:.2f}%"
)

PTP accounts: 13532
PTP accounts with successful payment: 5920
PTP → payment conversion: 43.75%


In [49]:

#  CAMPAIGN MASTER


campaigns_analysis = datasets["campaigns"].copy()

print("Campaign records:", len(campaigns_analysis))
print("Unique campaigns:", campaigns_analysis["campaign_id"].nunique())

display(campaigns_analysis)

Campaign records: 120
Unique campaigns: 120


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,2026-04-28 06:15:30,DPD>=60,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,2026-05-04 04:16:21,DPD>=60,2026-07-08 04:16:21
...,...,...,...,...,...,...,...
115,CMP0000116,DIGITAL_FOLLOWUP,VOICE,v1,2026-05-09 23:11:42,HIGH_RISK,2026-06-03 23:11:42
116,CMP0000117,BOUNCE,VOICE,v3,2026-02-25 19:12:52,HIGH_RISK,2026-04-09 19:12:52
117,CMP0000118,30DPD_W1,FIELD,v3,2026-03-07 10:51:06,DPD>=60,2026-05-11 10:51:06
118,CMP0000119,30DPD_W1,SMS,v2,2026-03-03 09:44:21,HIGH_RISK,2026-04-19 09:44:21


In [50]:

#  CAMPAIGN TARGETING VOLUME


campaign_targeting = (
    targeting
    .groupby("campaign_id", dropna=False)
    .agg(
        targeted_records=("target_id", "nunique"),
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

campaign_targeting

,campaign_id,targeted_records,targeted_accounts
0,CMP0000001,381,378
1,CMP0000002,373,371
2,CMP0000003,387,384
3,CMP0000004,365,364
4,CMP0000005,396,394
...,...,...,...
115,CMP0000116,371,368
116,CMP0000117,359,356
117,CMP0000118,346,344
118,CMP0000119,390,389


In [51]:

#  CAMPAIGN CALL ACTIVITY


campaign_calls = (
    calls_analysis
    .groupby("campaign_id", dropna=False)
    .agg(
        calls=("call_id", "nunique"),
        called_accounts=("account_id", "nunique"),
        total_duration_sec=("duration_sec", "sum")
    )
    .reset_index()
)

campaign_calls

,campaign_id,calls,called_accounts,total_duration_sec
0,CMP0000001,732,723,330276
1,CMP0000002,755,746,350729
2,CMP0000003,730,719,332469
3,CMP0000004,736,727,331660
4,CMP0000005,768,759,349802
...,...,...,...,...
115,CMP0000116,815,804,372412
116,CMP0000117,752,744,338674
117,CMP0000118,732,722,337591
118,CMP0000119,760,755,336620


In [52]:
# ============================================================
# 46. CAMPAIGN-ATTRIBUTED SUCCESSFUL PAYMENTS
# ============================================================

payments_attr = datasets["payments"].copy()
calls_attr = datasets["calls"].copy()

# Create separate timestamp columns BEFORE merging
payments_attr["payment_timestamp"] = pd.to_datetime(
    payments_attr["event_at"],
    errors="coerce"
)

calls_attr["call_timestamp"] = pd.to_datetime(
    calls_attr["event_at"],
    errors="coerce"
)

# Successful payments with valid timestamps
payment_events = payments_attr[
    payments_attr["payment_status"].eq("SUCCESS")
    & payments_attr["payment_timestamp"].notna()
].copy()

# Calls with valid timestamps
call_events = calls_attr[
    calls_attr["call_timestamp"].notna()
].copy()

# IMPORTANT: merge_asof requires the time columns to be sorted
payment_events = payment_events.sort_values(
    "payment_timestamp"
).reset_index(drop=True)

call_events = call_events.sort_values(
    "call_timestamp"
).reset_index(drop=True)

# Find the latest call before each successful payment
attribution_base = pd.merge_asof(
    payment_events,
    call_events,
    by="account_id",
    left_on="payment_timestamp",
    right_on="call_timestamp",
    direction="backward",
    suffixes=("_payment", "_call")
)

# Calculate hours between call and payment
attribution_base["hours_since_last_call"] = (
    attribution_base["payment_timestamp"]
    - attribution_base["call_timestamp"]
).dt.total_seconds() / 3600

# Keep calls occurring before payment and within 7 days
attribution_base = attribution_base[
    (attribution_base["hours_since_last_call"] >= 0)
    & (attribution_base["hours_since_last_call"] <= 168)
].copy()

# Keep only records with a campaign
attribution_base = attribution_base[
    attribution_base["campaign_id"].notna()
].copy()

print(
    "Successful payments with a campaign-attributed call:",
    len(attribution_base)
)

display(
    attribution_base[
        [
            "payment_id",
            "account_id",
            "payment_timestamp",
            "call_timestamp",
            "hours_since_last_call",
            "campaign_id",
            "amount"
        ]
    ].head(20)
)

Successful payments with a campaign-attributed call: 1608


,payment_id,account_id,payment_timestamp,call_timestamp,hours_since_last_call,campaign_id,amount
189,PAYMENT0022716,ACC0029551,2026-01-03 07:31:50,2026-01-02 10:19:24,21.207222,CMP0000092,7440.49
207,PAYMENT0022435,ACC0029560,2026-01-03 12:04:30,2026-01-02 15:18:22,20.768889,CMP0000006,43887.48
257,PAYMENT0022888,ACC0014362,2026-01-04 03:30:16,2026-01-01 05:30:15,70.000278,CMP0000006,148463.45
285,PAYMENT0007973,ACC0010164,2026-01-04 17:19:58,2026-01-01 20:04:59,69.249722,CMP0000025,94087.39
315,PAYMENT0010185,ACC0005945,2026-01-05 03:31:19,2026-01-01 14:21:36,85.161944,CMP0000036,62796.57
316,PAYMENT0011017,ACC0023889,2026-01-05 03:55:05,2026-01-01 10:43:17,89.196667,CMP0000049,29542.52
334,PAYMENT0020928,ACC0018862,2026-01-05 09:39:24,2026-01-03 19:47:00,37.873333,CMP0000106,130553.72
347,PAYMENT0013116,ACC0007107,2026-01-05 13:13:45,2026-01-05 12:48:00,0.429167,CMP0000028,10149.66
370,PAYMENT0020967,ACC0022175,2026-01-05 17:20:19,2026-01-03 13:54:27,51.431111,CMP0000047,3060.39
379,PAYMENT0024318,ACC0004585,2026-01-05 18:30:45,2026-01-02 15:12:52,75.298056,CMP0000043,55956.16


In [53]:
campaign_payments = (
    attribution_base
    .groupby("campaign_id")
    .agg(
        attributed_successful_payments=("payment_id", "nunique"),
        attributed_recovery=("amount", "sum"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(campaign_payments)

,campaign_id,attributed_successful_payments,attributed_recovery,paying_accounts
0,CMP0000001,15,1265943.03,15
1,CMP0000002,15,1434324.07,15
2,CMP0000003,10,779942.68,10
3,CMP0000004,11,1255758.48,11
4,CMP0000005,17,1159349.90,17
...,...,...,...,...
115,CMP0000116,15,1130952.09,15
116,CMP0000117,19,1876268.32,18
117,CMP0000118,13,968386.60,12
118,CMP0000119,13,1152544.15,13


In [54]:
# ============================================================
# 47. CAMPAIGN PERFORMANCE
# ============================================================

# Load fresh copies from the company-provided raw data
campaigns_analysis = datasets["campaigns"].copy()
targeting_analysis = datasets["daily_targeting"].copy()
calls_analysis = datasets["calls"].copy()

# ------------------------------------------------------------
# 1. Campaign master
# ------------------------------------------------------------

campaign_master = campaigns_analysis[
    [
        "campaign_id",
        "campaign_name",
        "channel",
        "strategy_version",
        "start_at",
        "end_at"
    ]
].drop_duplicates(
    subset=["campaign_id"]
)

# ------------------------------------------------------------
# 2. Targeting metrics
# ------------------------------------------------------------

campaign_targeting = (
    targeting_analysis
    .groupby("campaign_id", dropna=False)
    .agg(
        targeted_records=("target_id", "nunique"),
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Call metrics
# ------------------------------------------------------------

campaign_calls = (
    calls_analysis
    .groupby("campaign_id", dropna=False)
    .agg(
        calls=("call_id", "nunique"),
        called_accounts=("account_id", "nunique"),
        total_duration_sec=("duration_sec", "sum")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. Combine campaign information
# ------------------------------------------------------------

campaign_performance = (
    campaign_master
    .merge(
        campaign_targeting,
        on="campaign_id",
        how="left"
    )
    .merge(
        campaign_calls,
        on="campaign_id",
        how="left"
    )
    .merge(
        campaign_payments,
        on="campaign_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 5. Fill metrics with zero where no activity exists
# ------------------------------------------------------------

numeric_columns = [
    "targeted_records",
    "targeted_accounts",
    "calls",
    "called_accounts",
    "total_duration_sec",
    "attributed_successful_payments",
    "attributed_recovery",
    "paying_accounts"
]

for col in numeric_columns:
    if col in campaign_performance.columns:
        campaign_performance[col] = (
            pd.to_numeric(
                campaign_performance[col],
                errors="coerce"
            )
            .fillna(0)
        )

# ------------------------------------------------------------
# 6. Campaign conversion metrics
# ------------------------------------------------------------

campaign_performance["payment_conversion_pct"] = np.where(
    campaign_performance["targeted_accounts"] > 0,
    (
        campaign_performance["paying_accounts"]
        / campaign_performance["targeted_accounts"]
        * 100
    ),
    0
)

campaign_performance["recovery_per_targeted_account"] = np.where(
    campaign_performance["targeted_accounts"] > 0,
    (
        campaign_performance["attributed_recovery"]
        / campaign_performance["targeted_accounts"]
    ),
    0
)

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print(
    "Campaign performance records:",
    len(campaign_performance)
)

display(
    campaign_performance[
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version",
            "targeted_accounts",
            "called_accounts",
            "paying_accounts",
            "attributed_recovery",
            "payment_conversion_pct",
            "recovery_per_targeted_account"
        ]
    ].sort_values(
        "attributed_recovery",
        ascending=False
    )
)

Campaign performance records: 120


,campaign_id,campaign_name,channel,strategy_version,targeted_accounts,called_accounts,paying_accounts,attributed_recovery,payment_conversion_pct,recovery_per_targeted_account
70,CMP0000071,30DPD_W1,MIXED,v3,362,707,20,2014395.25,5.524862,5564.627762
116,CMP0000117,BOUNCE,VOICE,v3,356,744,18,1876268.32,5.056180,5270.416629
46,CMP0000047,BOUNCE,WHATSAPP,v2,379,756,19,1631284.56,5.013193,4304.180897
94,CMP0000095,DIGITAL_FOLLOWUP,VOICE,legacy,384,752,14,1611801.73,3.645833,4197.400339
32,CMP0000033,DIGITAL_FOLLOWUP,WHATSAPP,legacy,351,751,19,1567885.83,5.413105,4466.911197
...,...,...,...,...,...,...,...,...,...,...
43,CMP0000044,NPA_RECOVERY,SMS,v1,376,727,7,562400.85,1.861702,1495.746941
45,CMP0000046,BOUNCE,WHATSAPP,v3,385,772,7,501754.33,1.818182,1303.258000
26,CMP0000027,BOUNCE,VOICE,legacy,390,739,8,462051.90,2.051282,1184.748462
29,CMP0000030,30DPD_W1,SMS,v2,386,755,7,408240.59,1.813472,1057.618109


In [55]:
# ============================================================
# 48. CHANNEL PERFORMANCE
# ============================================================

channel_performance = (
    campaign_performance
    .groupby("channel", dropna=False)
    .agg(
        campaigns=("campaign_id", "nunique"),
        targeted_accounts=("targeted_accounts", "sum"),
        called_accounts=("called_accounts", "sum"),
        paying_accounts=("paying_accounts", "sum"),
        attributed_recovery=("attributed_recovery", "sum")
    )
    .reset_index()
)

channel_performance["payment_conversion_pct"] = np.where(
    channel_performance["targeted_accounts"] > 0,
    (
        channel_performance["paying_accounts"]
        / channel_performance["targeted_accounts"]
        * 100
    ),
    0
)

channel_performance["recovery_per_targeted_account"] = np.where(
    channel_performance["targeted_accounts"] > 0,
    (
        channel_performance["attributed_recovery"]
        / channel_performance["targeted_accounts"]
    ),
    0
)

display(channel_performance)

,channel,campaigns,targeted_accounts,called_accounts,paying_accounts,attributed_recovery,payment_conversion_pct,recovery_per_targeted_account
0,FIELD,18,6736,13258,241,18963839.80,3.577791,2815.296882
1,MIXED,23,8696,16888,308,23636789.76,3.541858,2718.122098
2,SMS,28,10476,20786,351,26119423.81,3.350515,2493.263059
3,VOICE,20,7455,14824,258,21270739.97,3.460765,2853.217970
4,WHATSAPP,31,11352,23109,403,31664496.60,3.550035,2789.331977


In [56]:
# ============================================================
# 49. SAVE CAMPAIGN & CHANNEL ANALYSIS
# ============================================================

campaign_performance.to_csv(
    PROCESSED_DIR / "campaign_performance.csv",
    index=False
)

channel_performance.to_csv(
    PROCESSED_DIR / "channel_performance.csv",
    index=False
)

print("Campaign analysis saved successfully.")

Campaign analysis saved successfully.


In [57]:
# ============================================================
# 50. AGENT & VENDOR DATA
# ============================================================

agents_analysis = datasets["agents"].copy()
vendor_telephony_analysis = datasets["vendor_telephony"].copy()
calls_analysis = datasets["calls"].copy()

print("Agents:", len(agents_analysis))
print("Vendor telephony:", len(vendor_telephony_analysis))

print("\nAgent columns:")
print(agents_analysis.columns.tolist())

print("\nVendor telephony columns:")
print(vendor_telephony_analysis.columns.tolist())

Agents: 30000
Vendor telephony: 15

Agent columns:
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']

Vendor telephony columns:
['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']


In [58]:
# ============================================================
# 51. AGENT CALL PERFORMANCE
# ============================================================

agent_calls = (
    calls_analysis
    .groupby("agent_id", dropna=False)
    .agg(
        calls=("call_id", "nunique"),
        accounts_called=("account_id", "nunique"),
        total_duration_sec=("duration_sec", "sum")
    )
    .reset_index()
)

agent_calls["avg_duration_sec"] = np.where(
    agent_calls["calls"] > 0,
    agent_calls["total_duration_sec"]
    / agent_calls["calls"],
    0
)

display(agent_calls)

,agent_id,calls,accounts_called,total_duration_sec,avg_duration_sec
0,AGT0000001,97,96,43453,447.969072
1,AGT0000002,94,94,41361,440.010638
2,AGT0000003,79,79,36414,460.936709
3,AGT0000004,87,87,37795,434.425287
4,AGT0000005,84,84,39583,471.226190
...,...,...,...,...,...
996,AGT0000997,91,91,45064,495.208791
997,AGT0000998,86,86,40328,468.930233
998,AGT0000999,79,79,37074,469.291139
999,AGT0001000,88,88,43161,490.465909


In [59]:
# ============================================================
# 52. AGENT PERFORMANCE WITH MASTER DATA
# ============================================================

agent_performance = (
    agents_analysis
    .merge(
        agent_calls,
        on="agent_id",
        how="left"
    )
)

for col in [
    "calls",
    "accounts_called",
    "total_duration_sec",
    "avg_duration_sec"
]:
    if col in agent_performance.columns:
        agent_performance[col] = (
            pd.to_numeric(
                agent_performance[col],
                errors="coerce"
            )
            .fillna(0)
        )

display(agent_performance)

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at,calls,accounts_called,total_duration_sec,avg_duration_sec
0,AGT0000760,EMP00323,Vikram Shah,VND0000012,T3,INACTIVE,2025-06-19 20:30:53,2025-12-11 03:40:22,99,98,51603,521.242424
1,AGT0000171,EMP00079,Pooja Nair,VND0000012,T1,INACTIVE,2025-08-26 11:06:26,2025-10-11 08:30:33,103,102,48216,468.116505
2,AGT0000766,EMP00030,Sneha Das,VND0000002,FIELD,SUSPENDED,2024-05-30 11:23:06,2026-05-16 03:02:02,96,96,44930,468.020833
3,AGT0000031,EMP00334,Aarav Sharma,VND0000007,T1,INACTIVE,2024-10-11 18:58:28,2025-10-16 13:31:21,94,94,44621,474.691489
4,AGT0000669,EMP00014,Aarav Sharma,VND0000001,T3,SUSPENDED,2024-09-01 19:55:06,2026-07-04 03:33:11,109,109,49197,451.348624
...,...,...,...,...,...,...,...,...,...,...,...,...
29995,AGT0000747,EMP00614,Rahul Verma,VND0000011,T1,SUSPENDED,2024-09-27 01:19:08,2025-08-22 21:41:38,99,98,41424,418.424242
29996,AGT0000734,EMP00827,Neha Singh,VND0000012,FIELD,ACTIVE,2024-03-22 00:55:18,2025-09-07 06:21:02,82,82,38132,465.024390
29997,AGT0000556,EMP00611,Priya Mehta,VND0000003,DIGITAL,ACTIVE,2024-08-03 01:30:39,2026-05-13 16:57:29,96,96,46761,487.093750
29998,AGT0000301,EMP00477,Rohan Patel,VND0000012,DIGITAL,SUSPENDED,2025-11-19 01:50:39,2025-08-03 15:39:53,90,90,39163,435.144444


In [60]:
# ============================================================
# 53. AGENT-ATTRIBUTED RECOVERY
# ============================================================

agent_recovery = (
    attribution_base[
        attribution_base["agent_id"].notna()
    ]
    .groupby("agent_id")
    .agg(
        attributed_successful_payments=("payment_id", "nunique"),
        attributed_recovery=("amount", "sum"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(agent_recovery)

,agent_id,attributed_successful_payments,attributed_recovery,paying_accounts
0,AGT0000001,1,142985.13,1
1,AGT0000002,3,159636.53,3
2,AGT0000003,2,151715.20,2
3,AGT0000004,4,397416.72,4
4,AGT0000006,2,88153.45,2
...,...,...,...,...
769,AGT0000994,1,82713.23,1
770,AGT0000995,2,143036.74,2
771,AGT0000996,1,41317.13,1
772,AGT0000997,1,88212.12,1


In [61]:
# ============================================================
# 54. FINAL AGENT PERFORMANCE
# ============================================================

agent_performance = (
    agent_performance
    .merge(
        agent_recovery,
        on="agent_id",
        how="left"
    )
)

for col in [
    "attributed_successful_payments",
    "attributed_recovery",
    "paying_accounts"
]:
    if col in agent_performance.columns:
        agent_performance[col] = (
            pd.to_numeric(
                agent_performance[col],
                errors="coerce"
            )
            .fillna(0)
        )

agent_performance["recovery_per_call"] = np.where(
    agent_performance["calls"] > 0,
    agent_performance["attributed_recovery"]
    / agent_performance["calls"],
    0
)

display(
    agent_performance.sort_values(
        "attributed_recovery",
        ascending=False
    )
)

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at,calls,accounts_called,total_duration_sec,avg_duration_sec,attributed_successful_payments,attributed_recovery,paying_accounts,recovery_per_call
17858,AGT0000730,EMP01037,Sneha Das,VND0000001,T1,ACTIVE,2024-12-31 02:12:45,2025-09-28 20:50:30,89,89,37559,422.011236,7.0,708177.02,7.0,7957.045169
4310,AGT0000730,EMP00440,Sneha Das,VND0000007,T1,INACTIVE,2025-05-05 01:28:16,2026-07-06 21:30:32,89,89,37559,422.011236,7.0,708177.02,7.0,7957.045169
17617,AGT0000730,EMP00831,Aarav Sharma,VND0000004,DIGITAL,INACTIVE,2024-03-12 05:34:26,2026-07-24 19:19:32,89,89,37559,422.011236,7.0,708177.02,7.0,7957.045169
29391,AGT0000730,EMP00443,Aarav Sharma,VND0000004,T1,ACTIVE,2024-01-05 07:38:22,2025-08-28 21:23:15,89,89,37559,422.011236,7.0,708177.02,7.0,7957.045169
5050,AGT0000730,EMP00267,Rohan Patel,VND0000015,T2,ACTIVE,2025-10-11 13:30:26,2025-12-14 23:54:03,89,89,37559,422.011236,7.0,708177.02,7.0,7957.045169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19831,AGT0000868,EMP00748,Rahul Verma,VND0000010,DIGITAL,ACTIVE,2024-03-08 00:34:55,2025-10-16 21:29:31,86,86,40424,470.046512,0.0,0.00,0.0,0.000000
16068,AGT0000558,EMP00109,Rohan Patel,VND0000010,T2,ACTIVE,2025-08-28 05:35:49,2025-10-03 14:10:27,79,79,38115,482.468354,0.0,0.00,0.0,0.000000
22361,AGT0000950,EMP00397,Vikram Shah,VND0000006,T2,ACTIVE,2024-03-02 21:49:49,2026-07-06 16:42:03,80,80,37856,473.200000,0.0,0.00,0.0,0.000000
25,AGT0000538,EMP01087,Neha Singh,VND0000001,FIELD,ACTIVE,2024-05-23 13:55:00,2025-07-03 04:22:08,94,94,45384,482.808511,0.0,0.00,0.0,0.000000


In [62]:
# ============================================================
# 55. VENDOR PERFORMANCE
# ============================================================

vendor_calls = (
    calls_analysis
    .groupby("vendor_id", dropna=False)
    .agg(
        calls=("call_id", "nunique"),
        accounts_called=("account_id", "nunique"),
        total_duration_sec=("duration_sec", "sum")
    )
    .reset_index()
)

vendor_recovery = (
    attribution_base[
        attribution_base["vendor_id"].notna()
    ]
    .groupby("vendor_id")
    .agg(
        attributed_successful_payments=("payment_id", "nunique"),
        attributed_recovery=("amount", "sum"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

vendor_performance = (
    vendor_calls
    .merge(
        vendor_recovery,
        on="vendor_id",
        how="left"
    )
)

vendor_performance[
    [
        "attributed_successful_payments",
        "attributed_recovery",
        "paying_accounts"
    ]
] = vendor_performance[
    [
        "attributed_successful_payments",
        "attributed_recovery",
        "paying_accounts"
    ]
].fillna(0)

vendor_performance["recovery_per_call"] = np.where(
    vendor_performance["calls"] > 0,
    vendor_performance["attributed_recovery"]
    / vendor_performance["calls"],
    0
)

display(
    vendor_performance.sort_values(
        "attributed_recovery",
        ascending=False
    )
)

,vendor_id,calls,accounts_called,total_duration_sec,attributed_successful_payments,attributed_recovery,paying_accounts,recovery_per_call
13,VND0000014,6041,5457,2781683,119,9625800.77,118,1593.411814
12,VND0000013,5904,5339,2710753,114,9272511.86,112,1570.547402
9,VND0000010,6163,5581,2834879,117,8897576.88,117,1443.708726
3,VND0000004,5925,5372,2713537,100,8676730.34,97,1464.427062
10,VND0000011,6072,5508,2788905,111,8636163.72,109,1422.293103
6,VND0000007,6069,5480,2740102,115,8601289.35,112,1417.249852
14,VND0000015,6066,5480,2797888,106,8473581.95,105,1396.897783
7,VND0000008,5911,5380,2689150,106,8112280.59,103,1372.404092
11,VND0000012,6010,5429,2755795,108,8026620.53,106,1335.544181
2,VND0000003,5964,5432,2758371,114,7982596.22,113,1338.463484


In [63]:
# ============================================================
# 56. SAVE AGENT & VENDOR PERFORMANCE
# ============================================================

agent_performance.to_csv(
    PROCESSED_DIR / "agent_performance.csv",
    index=False
)

vendor_performance.to_csv(
    PROCESSED_DIR / "vendor_performance.csv",
    index=False
)

print("Agent and vendor performance saved successfully.")

Agent and vendor performance saved successfully.


In [64]:
# ============================================================
# 57. 11% RECOVERY CLAIM TEST
# ============================================================

reported_recovery_rate = 11.0

calculated_recovery_rate = recovery_rate

difference_percentage_points = (
    calculated_recovery_rate
    - reported_recovery_rate
)

claim_test = pd.DataFrame([
    {
        "metric": "Reported Recovery Rate",
        "value_pct": reported_recovery_rate
    },
    {
        "metric": "Calculated Recovery Rate",
        "value_pct": calculated_recovery_rate
    },
    {
        "metric": "Difference",
        "value_pct": difference_percentage_points
    }
])

display(claim_test)

,metric,value_pct
0,Reported Recovery Rate,11.000000
1,Calculated Recovery Rate,12.312505
2,Difference,1.312505


In [65]:
# ============================================================
# 60. PAYMENT COVERAGE OF ACCOUNT PORTFOLIO
# ============================================================

total_accounts = accounts["account_id"].nunique()

successful_payment_accounts = (
    payments.loc[
        payments["payment_status"].eq("SUCCESS"),
        "account_id"
    ]
    .dropna()
    .nunique()
)

payment_account_coverage_pct = (
    successful_payment_accounts
    / total_accounts
    * 100
)

payment_coverage = pd.DataFrame([
    {
        "metric": "Total accounts",
        "value": total_accounts
    },
    {
        "metric": "Accounts with successful payment",
        "value": successful_payment_accounts
    },
    {
        "metric": "Account payment coverage",
        "value": payment_account_coverage_pct
    }
])

display(payment_coverage)

,metric,value
0,Total accounts,30000.000000
1,Accounts with successful payment,13109.000000
2,Account payment coverage,43.696667


In [66]:
# ============================================================
# 61. SAVE 11% CLAIM TEST
# ============================================================

claim_test_output = pd.DataFrame([{
    "reported_recovery_rate_pct": reported_recovery_rate,
    "calculated_recovery_rate_pct": calculated_recovery_rate,
    "difference_percentage_points": difference_percentage_points,
    "total_outstanding": total_outstanding,
    "recovered_amount": recovered_amount,
    "successful_payment_accounts": successful_payment_accounts,
    "total_accounts": total_accounts,
    "payment_account_coverage_pct": payment_account_coverage_pct
}])

claim_test_output.to_csv(
    PROCESSED_DIR / "recovery_claim_test.csv",
    index=False
)

display(claim_test_output)

,reported_recovery_rate_pct,calculated_recovery_rate_pct,difference_percentage_points,total_outstanding,recovered_amount,successful_payment_accounts,total_accounts,payment_account_coverage_pct
0,11.0,12.312505,1.312505,1.048904e+10,1.291463e+09,13109,30000,43.696667


In [67]:
# ============================================================
# 62. PREPARE MONTHLY PAYMENT DATA
# ============================================================

payment_mix = payments.merge(
    accounts[
        [
            "account_id",
            "dpd_bucket",
            "risk_segment",
            "outstanding_amount"
        ]
    ],
    on="account_id",
    how="left",
    validate="many_to_one"
)

payment_mix["event_at_dt"] = pd.to_datetime(
    payment_mix["event_at"],
    errors="coerce"
)

payment_mix["payment_month"] = (
    payment_mix["event_at_dt"]
    .dt.to_period("M")
)

print("Payment rows after account linkage:", len(payment_mix))

assert len(payment_mix) == len(payments)

Payment rows after account linkage: 24528


In [68]:
# ============================================================
# 63. MONTHLY RECOVERY BY PORTFOLIO MIX
# ============================================================

monthly_mix_recovery = (
    payment_mix[
        payment_mix["payment_status"].eq("SUCCESS")
        & payment_mix["payment_month"].notna()
    ]
    .groupby(
        [
            "payment_month",
            "dpd_bucket",
            "risk_segment"
        ],
        dropna=False
    )
    .agg(
        recovered_amount=("amount", "sum"),
        successful_payments=("payment_id", "nunique"),
        paying_accounts=("account_id", "nunique")
    )
    .reset_index()
)

display(
    monthly_mix_recovery.head(30)
)

,payment_month,dpd_bucket,risk_segment,recovered_amount,successful_payments,paying_accounts
0,2026-01,0-30,HIGH,21704382.94,283,273
1,2026-01,0-30,LOW,20892275.15,271,261
2,2026-01,0-30,MEDIUM,20017852.27,267,260
3,2026-01,0-30,NPA,20365427.29,270,255
4,2026-01,120+,HIGH,3669707.59,47,45
5,2026-01,120+,LOW,3735907.34,52,51
6,2026-01,120+,MEDIUM,4310176.68,58,56
7,2026-01,120+,NPA,3299418.03,45,43
8,2026-01,31-60,HIGH,9389271.94,120,116
9,2026-01,31-60,LOW,8476358.44,115,114


In [69]:
# ============================================================
# 64. PORTFOLIO MIX BY DPD × RISK
# ============================================================

portfolio_mix = (
    accounts
    .groupby(
        [
            "dpd_bucket",
            "risk_segment"
        ],
        dropna=False
    )
    .agg(
        accounts=("account_id", "nunique"),
        outstanding_amount=("outstanding_amount", "sum")
    )
    .reset_index()
)

portfolio_mix["portfolio_share_pct"] = (
    portfolio_mix["outstanding_amount"]
    / portfolio_mix["outstanding_amount"].sum()
    * 100
)

display(portfolio_mix)

,dpd_bucket,risk_segment,accounts,outstanding_amount,portfolio_share_pct
0,0-30,HIGH,3452,1.208611e+09,11.522617
1,0-30,LOW,3281,1.159493e+09,11.054330
2,0-30,MEDIUM,3514,1.231405e+09,11.739929
3,0-30,NPA,3318,1.182810e+09,11.276631
4,120+,HIGH,681,2.357435e+08,2.247524
5,120+,LOW,727,2.527330e+08,2.409497
6,120+,MEDIUM,618,2.033459e+08,1.938652
7,120+,NPA,668,2.285853e+08,2.179279
8,31-60,HIGH,1345,4.698857e+08,4.479780
9,31-60,LOW,1431,4.987723e+08,4.755178


In [70]:
# ============================================================
# 65. RECOVERY SHARE VS PORTFOLIO SHARE
# ============================================================

recovery_mix = (
    monthly_mix_recovery
    .groupby(
        ["dpd_bucket", "risk_segment"],
        dropna=False
    )
    .agg(
        recovered_amount=("recovered_amount", "sum"),
        successful_payments=("successful_payments", "sum")
    )
    .reset_index()
)

recovery_mix["recovery_share_pct"] = (
    recovery_mix["recovered_amount"]
    / recovery_mix["recovered_amount"].sum()
    * 100
)

mix_comparison = portfolio_mix.merge(
    recovery_mix,
    on=["dpd_bucket", "risk_segment"],
    how="left"
)

mix_comparison[
    [
        "recovered_amount",
        "successful_payments"
    ]
] = mix_comparison[
    [
        "recovered_amount",
        "successful_payments"
    ]
].fillna(0)

mix_comparison["recovery_minus_portfolio_share_pp"] = (
    mix_comparison["recovery_share_pct"]
    - mix_comparison["portfolio_share_pct"]
)

display(
    mix_comparison.sort_values(
        "recovered_amount",
        ascending=False
    )
)

,dpd_bucket,risk_segment,accounts,outstanding_amount,portfolio_share_pct,recovered_amount,successful_payments,recovery_share_pct,recovery_minus_portfolio_share_pp
0,0-30,HIGH,3452,1.208611e+09,11.522617,1.505187e+08,1986,11.654900,0.132283
2,0-30,MEDIUM,3514,1.231405e+09,11.739929,1.455536e+08,1982,11.270445,-0.469484
1,0-30,LOW,3281,1.159493e+09,11.054330,1.418773e+08,1915,10.985779,-0.068552
3,0-30,NPA,3318,1.182810e+09,11.276631,1.405744e+08,1861,10.884896,-0.391735
9,31-60,LOW,1431,4.987723e+08,4.755178,6.485815e+07,873,5.022068,0.266890
11,31-60,NPA,1379,4.711525e+08,4.491858,6.166359e+07,830,4.774708,0.282850
14,61-90,MEDIUM,1363,4.809616e+08,4.585375,6.132110e+07,798,4.748188,0.162814
8,31-60,HIGH,1345,4.698857e+08,4.479780,6.077404e+07,777,4.705829,0.226049
10,31-60,MEDIUM,1359,4.746798e+08,4.525486,6.033904e+07,801,4.672146,0.146660
12,61-90,HIGH,1364,4.779102e+08,4.556284,5.881050e+07,775,4.553789,-0.002494


In [71]:
# ============================================================
# 66. STANDARDIZED MONTHLY RECOVERY VIEW
# ============================================================

standardized_monthly = (
    monthly_recovery.copy()
)

standardized_monthly["standardized_recovery_rate_pct"] = (
    standardized_monthly["recovered_amount"]
    / total_outstanding
    * 100
)

display(standardized_monthly)

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct,payment_count_change,payment_count_change_pct,avg_successful_payment,standardized_recovery_rate_pct
0,2026-01,2413,1.833250e+08,NaN,NaN,NaN,NaN,75973.879304,1.747777
1,2026-02,2223,1.664619e+08,-1.686303e+07,-9.198435,-190.0,-7.874016,74881.665308,1.587009
2,2026-03,2475,1.851473e+08,1.868533e+07,11.224984,252.0,11.336032,74806.977321,1.765151
3,2026-04,2358,1.720308e+08,-1.311643e+07,-7.084320,-117.0,-4.727273,72956.252455,1.640102
4,2026-05,2413,1.816228e+08,9.591980e+06,5.575733,55.0,2.332485,75268.472018,1.731549
5,2026-06,2325,1.729824e+08,-8.640417e+06,-4.757341,-88.0,-3.646913,74401.034951,1.649174
6,2026-07,2397,1.842168e+08,1.123442e+07,6.494544,72.0,3.096774,76853.076813,1.756280
7,2026-08,595,4.567592e+07,-1.385409e+08,-75.205347,-1802.0,-75.177305,76766.256622,0.435464


In [72]:
# ============================================================
# 67. FEBRUARY → MARCH RECOVERY COMPARISON
# ============================================================

feb_mar = standardized_monthly[
    standardized_monthly["month"].isin([
        "2026-02",
        "2026-03"
    ])
].copy()

display(feb_mar)

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct,payment_count_change,payment_count_change_pct,avg_successful_payment,standardized_recovery_rate_pct
1,2026-02,2223,1.664619e+08,-16863028.78,-9.198435,-190.0,-7.874016,74881.665308,1.587009
2,2026-03,2475,1.851473e+08,18685326.89,11.224984,252.0,11.336032,74806.977321,1.765151


In [73]:
# ============================================================
# 68. FEBRUARY → MARCH CHANGE
# ============================================================

if set(feb_mar["month"]) == {"2026-02", "2026-03"}:

    feb_amount = feb_mar.loc[
        feb_mar["month"] == "2026-02",
        "recovered_amount"
    ].iloc[0]

    mar_amount = feb_mar.loc[
        feb_mar["month"] == "2026-03",
        "recovered_amount"
    ].iloc[0]

    feb_mar_change_pct = (
        (mar_amount - feb_amount)
        / feb_amount
        * 100
    )

    print(
        f"February recovery: ₹{feb_amount:,.2f}"
    )

    print(
        f"March recovery: ₹{mar_amount:,.2f}"
    )

    print(
        f"February → March change: "
        f"{feb_mar_change_pct:+.2f}%"
    )

else:
    print(
        "February and/or March is not available "
        "in the current payment data."
    )

February recovery: ₹166,461,941.98
March recovery: ₹185,147,268.87
February → March change: +11.22%


In [74]:
# ============================================================
# 69. SAVE MIX ANALYSIS
# ============================================================

mix_comparison.to_csv(
    PROCESSED_DIR / "recovery_mix_comparison.csv",
    index=False
)

standardized_monthly.to_csv(
    PROCESSED_DIR / "standardized_monthly_recovery.csv",
    index=False
)

print("Mix-adjusted analysis saved successfully.")

Mix-adjusted analysis saved successfully.


In [75]:
# ============================================================
# 70. PTP DATA STRUCTURE
# ============================================================

ptp_analysis = datasets["promises_to_pay"].copy()

print("PTP rows:", len(ptp_analysis))
print("PTP columns:")
print(ptp_analysis.columns.tolist())

display(ptp_analysis.head(20))

PTP rows: 18000
PTP columns:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS
5,PTP0000006,ACC0020223,BRW0004429,2026-06-14 00:43:18,AGT0000243,51103.49,2026-06-19 00:43:18,BROKEN,WHATSAPP
6,PTP0000007,ACC0018006,BRW0008673,2026-06-04 12:43:41,AGT0000973,18575.15,2026-06-24 12:43:41,CANCELLED,CALL
7,PTP0000008,ACC0011737,BRW0007288,2026-05-17 14:25:29,AGT0000044,64326.80,2026-05-31 14:25:29,OPEN,CALL
8,PTP0000009,ACC0021229,BRW0007682,2026-08-02 10:41:38,AGT0000149,56608.95,2026-08-15 10:41:38,BROKEN,WHATSAPP
9,PTP0000010,ACC0028901,BRW0009589,2026-02-05 15:34:26,AGT0000110,55403.24,2026-02-20 15:34:26,KEPT,SMS


In [76]:
# ============================================================
# 71. PTP TIMESTAMPS
# ============================================================

for col in ["event_at", "due_at", "created_at", "updated_at"]:
    if col in ptp_analysis.columns:
        ptp_analysis[f"{col}_dt"] = pd.to_datetime(
            ptp_analysis[col],
            errors="coerce"
        )

print("Timestamp columns prepared.")

display(ptp_analysis.head())

Timestamp columns prepared.


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source,event_at_dt
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD,2026-04-18 21:11:02
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP,2026-07-08 21:49:28
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS,2026-08-08 08:57:34
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP,2026-01-11 06:04:20
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS,2026-05-08 01:42:07


In [77]:
# ============================================================
# 72. PTP → SUCCESSFUL PAYMENT MATCH
# ============================================================

successful_payment_events = payments[
    payments["payment_status"].eq("SUCCESS")
].copy()

successful_payment_events["payment_timestamp"] = pd.to_datetime(
    successful_payment_events["event_at"],
    errors="coerce"
)

ptp_analysis["ptp_timestamp"] = pd.to_datetime(
    ptp_analysis["event_at"],
    errors="coerce"
)

# Keep valid timestamps
ptp_valid = ptp_analysis[
    ptp_analysis["ptp_timestamp"].notna()
].copy()

payment_valid = successful_payment_events[
    successful_payment_events["payment_timestamp"].notna()
].copy()

# Sort by account and timestamp
ptp_valid = ptp_valid.sort_values(
    ["account_id", "ptp_timestamp"]
).reset_index(drop=True)

payment_valid = payment_valid.sort_values(
    ["account_id", "payment_timestamp"]
).reset_index(drop=True)

print("Valid PTP records:", len(ptp_valid))
print("Valid successful payments:", len(payment_valid))

Valid PTP records: 18000
Valid successful payments: 17210


In [78]:
# ============================================================
# 73. MATCH SUCCESSFUL PAYMENT AFTER PTP
# ============================================================

# Work from clean copies
ptp_match = ptp_valid.copy()
payment_match = payment_valid.copy()

# IMPORTANT:
# pandas merge_asof requires the time columns to be globally sorted
ptp_match = ptp_match.sort_values(
    "ptp_timestamp"
).reset_index(drop=True)

payment_match = payment_match.sort_values(
    "payment_timestamp"
).reset_index(drop=True)

# Find the first successful payment after each PTP
ptp_payment_match = pd.merge_asof(
    ptp_match,
    payment_match[
        [
            "account_id",
            "payment_id",
            "payment_timestamp",
            "amount"
        ]
    ],
    by="account_id",
    left_on="ptp_timestamp",
    right_on="payment_timestamp",
    direction="forward",
    tolerance=pd.Timedelta(days=30)
)

# Calculate time from PTP to payment
ptp_payment_match["hours_to_payment"] = (
    ptp_payment_match["payment_timestamp"]
    - ptp_payment_match["ptp_timestamp"]
).dt.total_seconds() / 3600

print(
    "PTP records matched:",
    len(ptp_payment_match)
)

display(
    ptp_payment_match[
        [
            "ptp_id",
            "account_id",
            "ptp_timestamp",
            "payment_timestamp",
            "hours_to_payment",
            "payment_id",
            "amount"
        ]
    ].head(20)
)

PTP records matched: 18000


,ptp_id,account_id,ptp_timestamp,payment_timestamp,hours_to_payment,payment_id,amount
0,PTP0001356,ACC0013190,2026-01-01 00:24:14,NaT,NaN,NaN,NaN
1,PTP0014736,ACC0016818,2026-01-01 00:25:26,NaT,NaN,NaN,NaN
2,PTP0014810,ACC0028863,2026-01-01 00:51:12,NaT,NaN,NaN,NaN
3,PTP0016511,ACC0006373,2026-01-01 00:52:16,NaT,NaN,NaN,NaN
4,PTP0016972,ACC0007009,2026-01-01 00:55:30,NaT,NaN,NaN,NaN
5,PTP0007919,ACC0014693,2026-01-01 00:55:41,NaT,NaN,NaN,NaN
6,PTP0004837,ACC0020497,2026-01-01 01:23:53,NaT,NaN,NaN,NaN
7,PTP0002619,ACC0009194,2026-01-01 01:34:41,NaT,NaN,NaN,NaN
8,PTP0017742,ACC0000103,2026-01-01 01:36:02,NaT,NaN,NaN,NaN
9,PTP0015047,ACC0026628,2026-01-01 01:54:39,NaT,NaN,NaN,NaN


In [79]:
# ============================================================
# 74. PTP FULFILLMENT FLAG
# ============================================================

ptp_payment_match["payment_after_ptp"] = (
    ptp_payment_match["payment_id"].notna()
    & (ptp_payment_match["hours_to_payment"] >= 0)
)

ptp_payment_match["payment_after_ptp"].value_counts(
    dropna=False
)

payment_after_ptp
False    16723
True      1277
Name: count, dtype: int64

In [80]:
# ============================================================
# 75. PTP FULFILLMENT SUMMARY
# ============================================================

total_valid_ptps = len(ptp_payment_match)

fulfilled_ptps = (
    ptp_payment_match["payment_after_ptp"].sum()
)

ptp_fulfillment_rate = (
    fulfilled_ptps
    / total_valid_ptps
    * 100
    if total_valid_ptps > 0
    else 0
)

ptp_fulfillment_summary = pd.DataFrame([
    {
        "metric": "Valid PTP records",
        "value": total_valid_ptps
    },
    {
        "metric": "PTPs followed by successful payment",
        "value": fulfilled_ptps
    },
    {
        "metric": "PTP fulfillment rate",
        "value": ptp_fulfillment_rate
    }
])

display(ptp_fulfillment_summary)

,metric,value
0,Valid PTP records,18000.000000
1,PTPs followed by successful payment,1277.000000
2,PTP fulfillment rate,7.094444


In [81]:
# ============================================================
# 76. PTP STATUS VS OBSERVED PAYMENT
# ============================================================

status_columns = [
    col for col in ptp_analysis.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "status",
            "outcome",
            "result"
        ]
    )
    and not col.endswith("_dt")
]

print("Potential PTP status/outcome columns:")
print(status_columns)

for col in status_columns:
    print(f"\n--- {col} ---")
    display(
        ptp_analysis[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="records")
    )

Potential PTP status/outcome columns:
['status']

--- status ---


,status,records
0,BROKEN,4553
1,CANCELLED,4543
2,KEPT,4489
3,OPEN,4415


In [82]:
# ============================================================
# 77. SAVE PTP INTEGRITY ANALYSIS
# ============================================================

ptp_payment_match.to_csv(
    PROCESSED_DIR / "ptp_payment_match.csv",
    index=False
)

ptp_fulfillment_summary.to_csv(
    PROCESSED_DIR / "ptp_fulfillment_summary.csv",
    index=False
)

print("PTP integrity analysis saved successfully.")

PTP integrity analysis saved successfully.


In [83]:
# ============================================================
# 78. PTP PERFORMANCE SUMMARY
# ============================================================

ptp_summary = (
    ptp_payment_match
    .groupby("status", dropna=False)
    .agg(
        ptp_records=("ptp_id", "nunique"),
        ptp_accounts=("account_id", "nunique"),
        promised_amount=("promised_amount", "sum"),
        successful_payments=("payment_after_ptp", "sum"),
        recovered_amount=("amount", "sum")
    )
    .reset_index()
)

ptp_summary["payment_conversion_pct"] = np.where(
    ptp_summary["ptp_accounts"] > 0,
    ptp_summary["successful_payments"]
    / ptp_summary["ptp_accounts"]
    * 100,
    0
)

display(
    ptp_summary.sort_values(
        "recovered_amount",
        ascending=False
    )
)

,status,ptp_records,ptp_accounts,promised_amount,successful_payments,recovered_amount,payment_conversion_pct
3,OPEN,4415,4093,2.219315e+08,326,25135494.66,7.964818
2,KEPT,4489,4164,2.249404e+08,336,24765764.63,8.069164
0,BROKEN,4553,4268,2.287966e+08,300,22362756.49,7.029053
1,CANCELLED,4543,4219,2.285406e+08,315,22348313.64,7.466224


In [84]:
# ============================================================
# 79. CONSOLIDATED RECOVERY EVIDENCE
# ============================================================

root_cause_summary = pd.DataFrame([
    {
        "analysis_area": "Overall Recovery",
        "metric": "Calculated Recovery Rate",
        "value": calculated_recovery_rate,
        "unit": "%"
    },
    {
        "analysis_area": "Overall Recovery",
        "metric": "Reported Recovery Rate",
        "value": reported_recovery_rate,
        "unit": "%"
    },
    {
        "analysis_area": "Overall Recovery",
        "metric": "Difference vs Reported",
        "value": difference_percentage_points,
        "unit": "percentage points"
    },
    {
        "analysis_area": "Portfolio",
        "metric": "Total Accounts",
        "value": total_accounts,
        "unit": "accounts"
    },
    {
        "analysis_area": "Portfolio",
        "metric": "Total Outstanding",
        "value": total_outstanding,
        "unit": "₹"
    },
    {
        "analysis_area": "Payments",
        "metric": "Recovered Amount",
        "value": recovered_amount,
        "unit": "₹"
    },
    {
        "analysis_area": "Payments",
        "metric": "Successful Payment Accounts",
        "value": successful_payment_accounts,
        "unit": "accounts"
    },
    {
        "analysis_area": "Payments",
        "metric": "Account Payment Coverage",
        "value": payment_account_coverage_pct,
        "unit": "%"
    },
    {
        "analysis_area": "PTP",
        "metric": "PTP Fulfillment Rate",
        "value": ptp_fulfillment_rate,
        "unit": "%"
    }
])

display(root_cause_summary)

,analysis_area,metric,value,unit
0,Overall Recovery,Calculated Recovery Rate,1.231251e+01,%
1,Overall Recovery,Reported Recovery Rate,1.100000e+01,%
2,Overall Recovery,Difference vs Reported,1.312505e+00,percentage points
3,Portfolio,Total Accounts,3.000000e+04,accounts
4,Portfolio,Total Outstanding,1.048904e+10,₹
5,Payments,Recovered Amount,1.291463e+09,₹
6,Payments,Successful Payment Accounts,1.310900e+04,accounts
7,Payments,Account Payment Coverage,4.369667e+01,%
8,PTP,PTP Fulfillment Rate,7.094444e+00,%


In [85]:
# ============================================================
# 80. SAVE ROOT-CAUSE EVIDENCE
# ============================================================

root_cause_summary.to_csv(
    PROCESSED_DIR / "root_cause_evidence.csv",
    index=False
)

print("Root-cause evidence saved successfully.")

Root-cause evidence saved successfully.


In [86]:
# ============================================================
# 81. ROOT-CAUSE ANALYSIS CHECKLIST
# ============================================================

root_cause_questions = pd.DataFrame([
    {
        "question": "Does calculated recovery match the reported 11%?",
        "analysis": "Overall recovery claim test"
    },
    {
        "question": "Is recovery concentrated in particular DPD buckets?",
        "analysis": "Recovery by DPD"
    },
    {
        "question": "Is recovery concentrated in particular risk segments?",
        "analysis": "Recovery by risk segment"
    },
    {
        "question": "Does portfolio mix explain recovery differences?",
        "analysis": "DPD × risk mix comparison"
    },
    {
        "question": "Which campaigns/channels are associated with recovery?",
        "analysis": "Campaign/channel performance"
    },
    {
        "question": "Are there meaningful differences across agents/vendors?",
        "analysis": "Agent/vendor performance"
    },
    {
        "question": "Are recorded PTPs followed by successful payments?",
        "analysis": "PTP integrity"
    },
    {
        "question": "Are data-quality issues affecting interpretation?",
        "analysis": "Data-quality findings"
    }
])

display(root_cause_questions)

,question,analysis
0,Does calculated recovery match the reported 11%?,Overall recovery claim test
1,Is recovery concentrated in particular DPD buc...,Recovery by DPD
2,Is recovery concentrated in particular risk se...,Recovery by risk segment
3,Does portfolio mix explain recovery differences?,DPD × risk mix comparison
4,Which campaigns/channels are associated with r...,Campaign/channel performance
5,Are there meaningful differences across agents...,Agent/vendor performance
6,Are recorded PTPs followed by successful payme...,PTP integrity
7,Are data-quality issues affecting interpretation?,Data-quality findings


In [87]:
# ============================================================
# 82. FINAL FINDINGS STRUCTURE
# ============================================================

final_findings = pd.DataFrame([
    {
        "finding_id": "F-01",
        "area": "Recovery",
        "finding": "Calculated recovery rate vs reported 11%",
        "evidence": f"Calculated recovery rate = {calculated_recovery_rate:.2f}%; reported rate = {reported_recovery_rate:.2f}%",
        "business_impact": "Determines whether the reported recovery figure is supported by the supplied data.",
        "recommendation": "Use the reconciled recovery definition consistently in management reporting."
    },
    {
        "finding_id": "F-02",
        "area": "Portfolio Mix",
        "finding": "Recovery differs across DPD and risk segments",
        "evidence": "See recovery_by_dpd.csv, recovery_by_risk_segment.csv and recovery_mix_comparison.csv.",
        "business_impact": "Portfolio composition can materially influence aggregate recovery.",
        "recommendation": "Track recovery using consistent DPD and risk-segment cohorts."
    },
    {
        "finding_id": "F-03",
        "area": "Campaigns",
        "finding": "Campaign performance varies by campaign and channel",
        "evidence": "See campaign_performance.csv and channel_performance.csv.",
        "business_impact": "Different collection strategies may be associated with different recovery outcomes.",
        "recommendation": "Compare campaigns using common targeting and attribution definitions before scaling a strategy."
    },
    {
        "finding_id": "F-04",
        "area": "Operations",
        "finding": "Agent and vendor performance varies",
        "evidence": "See agent_performance.csv and vendor_performance.csv.",
        "business_impact": "Operational execution may contribute to differences in collection outcomes.",
        "recommendation": "Use normalized productivity and recovery metrics when evaluating operational performance."
    },
    {
        "finding_id": "F-05",
        "area": "PTP",
        "finding": "Recorded PTPs should be validated against subsequent successful payments",
        "evidence": f"Observed PTP fulfillment rate = {ptp_fulfillment_rate:.2f}%.",
        "business_impact": "PTP volume alone may overstate actual collection effectiveness.",
        "recommendation": "Track PTP fulfillment using an independently matched payment outcome."
    },
    {
        "finding_id": "F-06",
        "area": "Data Quality",
        "finding": "Data-quality and attribution limitations affect interpretation",
        "evidence": "See the data-quality log and attribution analysis.",
        "business_impact": "Unresolved data issues can distort recovery and operational comparisons.",
        "recommendation": "Maintain explicit data-quality flags and metric definitions in recurring reporting."
    }
])

display(final_findings)

,finding_id,area,finding,evidence,business_impact,recommendation
0,F-01,Recovery,Calculated recovery rate vs reported 11%,Calculated recovery rate = 12.31%; reported ra...,Determines whether the reported recovery figur...,Use the reconciled recovery definition consist...
1,F-02,Portfolio Mix,Recovery differs across DPD and risk segments,"See recovery_by_dpd.csv, recovery_by_risk_segm...",Portfolio composition can materially influence...,Track recovery using consistent DPD and risk-s...
2,F-03,Campaigns,Campaign performance varies by campaign and ch...,See campaign_performance.csv and channel_perfo...,Different collection strategies may be associa...,Compare campaigns using common targeting and a...
3,F-04,Operations,Agent and vendor performance varies,See agent_performance.csv and vendor_performan...,Operational execution may contribute to differ...,Use normalized productivity and recovery metri...
4,F-05,PTP,Recorded PTPs should be validated against subs...,Observed PTP fulfillment rate = 7.09%.,PTP volume alone may overstate actual collecti...,Track PTP fulfillment using an independently m...
5,F-06,Data Quality,Data-quality and attribution limitations affec...,See the data-quality log and attribution analy...,Unresolved data issues can distort recovery an...,Maintain explicit data-quality flags and metri...


In [88]:
# ============================================================
# 83. SAVE FINAL FINDINGS
# ============================================================

final_findings.to_csv(
    PROCESSED_DIR / "final_findings.csv",
    index=False
)

print("Final findings saved successfully.")

Final findings saved successfully.


In [89]:
# ============================================================
# 84. EXECUTIVE KPI SNAPSHOT
# ============================================================

executive_snapshot = pd.DataFrame([{
    "total_accounts": total_accounts,
    "total_outstanding": total_outstanding,
    "recovered_amount": recovered_amount,
    "calculated_recovery_rate_pct": calculated_recovery_rate,
    "reported_recovery_rate_pct": reported_recovery_rate,
    "difference_vs_reported_pp": difference_percentage_points,
    "successful_payment_accounts": successful_payment_accounts,
    "payment_account_coverage_pct": payment_account_coverage_pct,
    "valid_ptp_records": total_valid_ptps,
    "ptp_fulfillment_rate_pct": ptp_fulfillment_rate
}])

display(executive_snapshot)

,total_accounts,total_outstanding,recovered_amount,calculated_recovery_rate_pct,reported_recovery_rate_pct,difference_vs_reported_pp,successful_payment_accounts,payment_account_coverage_pct,valid_ptp_records,ptp_fulfillment_rate_pct
0,30000,1.048904e+10,1.291463e+09,12.312505,11.0,1.312505,13109,43.696667,18000,7.094444


In [90]:
# ============================================================
# 85. SAVE EXECUTIVE SNAPSHOT
# ============================================================

executive_snapshot.to_csv(
    PROCESSED_DIR / "executive_snapshot.csv",
    index=False
)

print("Executive snapshot saved successfully.")

Executive snapshot saved successfully.


In [91]:
# ============================================================
# 86. FINAL OUTPUT AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

# Set project paths for this notebook/kernel
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)

expected_outputs = [
    "recovery_summary.csv",
    "monthly_recovery.csv",
    "core_kpis.csv",
    "recovery_by_dpd.csv",
    "recovery_by_risk_segment.csv",
    "recovery_by_loan_type.csv",
    "recovery_by_dpd_risk.csv",
    "campaign_performance.csv",
    "channel_performance.csv",
    "agent_performance.csv",
    "vendor_performance.csv",
    "ptp_payment_match.csv",
    "ptp_fulfillment_summary.csv",
    "root_cause_evidence.csv",
    "final_findings.csv",
    "executive_snapshot.csv",
    "recovery_claim_test.csv",
    "recovery_mix_comparison.csv",
    "standardized_monthly_recovery.csv"
]

print("\nOUTPUT AUDIT")
print("=" * 70)

missing_outputs = []
existing_outputs = []

for filename in expected_outputs:
    filepath = PROCESSED_DIR / filename

    if filepath.exists():
        existing_outputs.append(filename)
        print(f"[OK]      {filename}")
    else:
        missing_outputs.append(filename)
        print(f"[MISSING] {filename}")

print("\n" + "=" * 70)
print(f"Expected outputs : {len(expected_outputs)}")
print(f"Existing outputs : {len(existing_outputs)}")
print(f"Missing outputs  : {len(missing_outputs)}")

Project root: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics
Processed directory: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed

OUTPUT AUDIT
[OK]      recovery_summary.csv
[OK]      monthly_recovery.csv
[OK]      core_kpis.csv
[OK]      recovery_by_dpd.csv
[OK]      recovery_by_risk_segment.csv
[OK]      recovery_by_loan_type.csv
[OK]      recovery_by_dpd_risk.csv
[OK]      campaign_performance.csv
[OK]      channel_performance.csv
[OK]      agent_performance.csv
[OK]      vendor_performance.csv
[OK]      ptp_payment_match.csv
[OK]      ptp_fulfillment_summary.csv
[OK]      root_cause_evidence.csv
[OK]      final_findings.csv
[OK]      executive_snapshot.csv
[OK]      recovery_claim_test.csv
[OK]      recovery_mix_comparison.csv
[OK]      standardized_monthly_recovery.csv

Expected outputs : 19
Existing outputs : 19
Missing outputs  : 0


In [92]:
# ============================================================
# 87. OUTPUT CONTENT AUDIT
# ============================================================

output_audit = []

for filename in existing_outputs:
    filepath = PROCESSED_DIR / filename

    try:
        df = pd.read_csv(filepath, low_memory=False)

        output_audit.append({
            "file": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "status": "OK"
        })

    except Exception as e:
        output_audit.append({
            "file": filename,
            "rows": None,
            "columns": None,
            "status": f"ERROR: {str(e)}"
        })

output_audit_df = pd.DataFrame(output_audit)

display(output_audit_df)

,file,rows,columns,status
0,recovery_summary.csv,1,5,OK
1,monthly_recovery.csv,8,8,OK
2,core_kpis.csv,5,2,OK
3,recovery_by_dpd.csv,5,9,OK
4,recovery_by_risk_segment.csv,4,9,OK
5,recovery_by_loan_type.csv,5,7,OK
6,recovery_by_dpd_risk.csv,20,5,OK
7,campaign_performance.csv,120,16,OK
8,channel_performance.csv,5,8,OK
9,agent_performance.csv,30000,16,OK


In [93]:
# ============================================================
# 88. SAVE OUTPUT AUDIT
# ============================================================

output_audit_df.to_csv(
    PROCESSED_DIR / "output_audit.csv",
    index=False
)

print("Output audit saved.")

Output audit saved.


In [94]:
# ============================================================
# 89. PROJECT STRUCTURE CHECK
# ============================================================

project_directories = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "dashboard",
    "reports",
    "architecture",
    "outputs"
]

print("PROJECT STRUCTURE")
print("=" * 70)

for directory in project_directories:
    path = PROJECT_ROOT / directory

    print(
        f"[{'OK' if path.exists() else 'MISSING'}] {directory}"
    )

PROJECT STRUCTURE
[OK] data/raw
[OK] data/processed
[OK] notebooks
[OK] sql
[OK] dashboard
[OK] reports
[OK] architecture
[OK] outputs


In [95]:
# ============================================================
# 90. DASHBOARD OUTPUT SETUP
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DASHBOARD_DIR = PROJECT_ROOT / "dashboard"

DASHBOARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Dashboard directory:", DASHBOARD_DIR)

Dashboard directory: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\dashboard


In [96]:
# ============================================================
# 91. LOAD EXECUTIVE SNAPSHOT
# ============================================================

executive_snapshot = pd.read_csv(
    PROCESSED_DIR / "executive_snapshot.csv"
)

display(executive_snapshot)

,total_accounts,total_outstanding,recovered_amount,calculated_recovery_rate_pct,reported_recovery_rate_pct,difference_vs_reported_pp,successful_payment_accounts,payment_account_coverage_pct,valid_ptp_records,ptp_fulfillment_rate_pct
0,30000,1.048904e+10,1.291463e+09,12.312505,11.0,1.312505,13109,43.696667,18000,7.094444


In [97]:
# ============================================================
# 92. LOAD MONTHLY RECOVERY
# ============================================================

monthly_recovery = pd.read_csv(
    PROCESSED_DIR / "monthly_recovery.csv"
)

display(monthly_recovery)

,month,successful_payment_count,recovered_amount,recovery_change,recovery_change_pct,payment_count_change,payment_count_change_pct,avg_successful_payment
0,2026-01,2413,1.833250e+08,NaN,NaN,NaN,NaN,75973.879304
1,2026-02,2223,1.664619e+08,-1.686303e+07,-9.198435,-190.0,-7.874016,74881.665308
2,2026-03,2475,1.851473e+08,1.868533e+07,11.224984,252.0,11.336032,74806.977321
3,2026-04,2358,1.720308e+08,-1.311643e+07,-7.084320,-117.0,-4.727273,72956.252455
4,2026-05,2413,1.816228e+08,9.591980e+06,5.575733,55.0,2.332485,75268.472018
5,2026-06,2325,1.729824e+08,-8.640417e+06,-4.757341,-88.0,-3.646913,74401.034951
6,2026-07,2397,1.842168e+08,1.123442e+07,6.494544,72.0,3.096774,76853.076813
7,2026-08,595,4.567592e+07,-1.385409e+08,-75.205347,-1802.0,-75.177305,76766.256622


In [98]:
# ============================================================
# 93. LOAD SEGMENTATION OUTPUTS
# ============================================================

recovery_by_dpd = pd.read_csv(
    PROCESSED_DIR / "recovery_by_dpd.csv"
)

recovery_by_risk = pd.read_csv(
    PROCESSED_DIR / "recovery_by_risk_segment.csv"
)

recovery_by_loan = pd.read_csv(
    PROCESSED_DIR / "recovery_by_loan_type.csv"
)

print("DPD rows:", len(recovery_by_dpd))
print("Risk rows:", len(recovery_by_risk))
print("Loan type rows:", len(recovery_by_loan))

DPD rows: 5
Risk rows: 4
Loan type rows: 5


In [99]:
# ============================================================
# 94. LOAD CAMPAIGN & CHANNEL OUTPUTS
# ============================================================

campaign_performance = pd.read_csv(
    PROCESSED_DIR / "campaign_performance.csv"
)

channel_performance = pd.read_csv(
    PROCESSED_DIR / "channel_performance.csv"
)

print("Campaign rows:", len(campaign_performance))
print("Channel rows:", len(channel_performance))

Campaign rows: 120
Channel rows: 5


In [100]:
# ============================================================
# 95. LOAD OPERATIONAL OUTPUTS
# ============================================================

agent_performance = pd.read_csv(
    PROCESSED_DIR / "agent_performance.csv"
)

vendor_performance = pd.read_csv(
    PROCESSED_DIR / "vendor_performance.csv"
)

ptp_summary = pd.read_csv(
    PROCESSED_DIR / "ptp_fulfillment_summary.csv"
)

print("Agent rows:", len(agent_performance))
print("Vendor rows:", len(vendor_performance))
print("PTP summary rows:", len(ptp_summary))

Agent rows: 30000
Vendor rows: 15
PTP summary rows: 3


In [101]:
# ============================================================
# 96. DASHBOARD DATA MANIFEST
# ============================================================

dashboard_manifest = pd.DataFrame([
    {
        "dataset": "executive_snapshot",
        "purpose": "Executive KPI cards and recovery claim"
    },
    {
        "dataset": "monthly_recovery",
        "purpose": "Recovery trend over time"
    },
    {
        "dataset": "recovery_by_dpd",
        "purpose": "Recovery by DPD bucket"
    },
    {
        "dataset": "recovery_by_risk_segment",
        "purpose": "Recovery by risk segment"
    },
    {
        "dataset": "recovery_by_loan_type",
        "purpose": "Recovery by loan type"
    },
    {
        "dataset": "campaign_performance",
        "purpose": "Campaign performance and attribution"
    },
    {
        "dataset": "channel_performance",
        "purpose": "Channel performance"
    },
    {
        "dataset": "agent_performance",
        "purpose": "Agent performance"
    },
    {
        "dataset": "vendor_performance",
        "purpose": "Vendor performance"
    },
    {
        "dataset": "ptp_fulfillment_summary",
        "purpose": "PTP integrity and fulfillment"
    }
])

display(dashboard_manifest)

,dataset,purpose
0,executive_snapshot,Executive KPI cards and recovery claim
1,monthly_recovery,Recovery trend over time
2,recovery_by_dpd,Recovery by DPD bucket
3,recovery_by_risk_segment,Recovery by risk segment
4,recovery_by_loan_type,Recovery by loan type
5,campaign_performance,Campaign performance and attribution
6,channel_performance,Channel performance
7,agent_performance,Agent performance
8,vendor_performance,Vendor performance
9,ptp_fulfillment_summary,PTP integrity and fulfillment


In [102]:
# ============================================================
# 97. SAVE DASHBOARD MANIFEST
# ============================================================

dashboard_manifest.to_csv(
    DASHBOARD_DIR / "dashboard_data_manifest.csv",
    index=False
)

print("Dashboard manifest saved successfully.")

Dashboard manifest saved successfully.


## 87. DUPLICATE PAYMENT INVESTIGATION

In [104]:
# ============================================================
# 87. DUPLICATE PAYMENT INVESTIGATION
# ============================================================

payment_forensics = payments.copy()

print("Total payment records:", len(payment_forensics))
print("Unique payment IDs:", payment_forensics["payment_id"].nunique())

# 1. Duplicate payment IDs
duplicate_payment_ids = (
    payment_forensics[
        payment_forensics["payment_id"].duplicated(keep=False)
    ]
    .sort_values("payment_id")
)

print("\nDuplicate payment ID records:", len(duplicate_payment_ids))
display(duplicate_payment_ids.head(20))

# 2. Duplicate payment references
if "payment_reference" in payment_forensics.columns:
    duplicate_references = (
        payment_forensics[
            payment_forensics["payment_reference"].duplicated(keep=False)
        ]
        .sort_values("payment_reference")
    )

    print(
        "\nDuplicate payment reference records:",
        len(duplicate_references)
    )
    display(duplicate_references.head(20))

# 3. Exact duplicate rows
exact_duplicates = payment_forensics[
    payment_forensics.duplicated(keep=False)
]

print("\nExact duplicate rows:", len(exact_duplicates))
display(exact_duplicates.head(20))

# 4. Duplicate payment IDs and financial impact
duplicate_id_summary = (
    duplicate_payment_ids
    .groupby("payment_id")
    .agg(
        records=("payment_id", "size"),
        accounts=("account_id", "nunique"),
        total_amount=("amount", "sum"),
        statuses=("payment_status", "nunique")
    )
    .reset_index()
)

print("\nDuplicate payment ID summary:")
display(duplicate_id_summary.head(20))

Total payment records: 24528
Unique payment IDs: 24514

Duplicate payment ID records: 28


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,quality_flag,event_at_dt,payment_month
1365,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,TXN0000046535,85205.16,SUCCESS,CASH,VND0000015,VALID,2026-05-14 07:52:59,2026-05
24514,PAYMENT0001390,ACC0001624,BRW0004200,2026-05-14 07:52:59,NaN,85205.16,SUCCESS,CASH,VND0000015,MISSING_REFERENCE,2026-05-14 07:52:59,2026-05
2054,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,TXN0000001302,5570.92,SUCCESS,NETBANKING,VND0000001,VALID,2026-07-01 09:19:16,2026-07
24515,PAYMENT0002098,ACC0023839,BRW0008653,2026-07-01 09:19:16,NaN,5570.92,SUCCESS,NETBANKING,VND0000001,MISSING_REFERENCE,2026-07-01 09:19:16,2026-07
2250,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,TXN0000067311,59429.86,SUCCESS,CARD,VND0000011,VALID,2026-06-15 03:09:47,2026-06
24516,PAYMENT0002303,ACC0021362,BRW0001779,2026-06-15 03:09:47,NaN,59429.86,SUCCESS,CARD,VND0000011,MISSING_REFERENCE,2026-06-15 03:09:47,2026-06
24527,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,TXN0000041890,107235.47,SUCCESS,NACH,VND0000002,VALID,2026-06-28 13:50:03,2026-06
5067,PAYMENT0005191,ACC0001901,BRW0001234,2026-06-28 13:50:03,NaN,107235.47,SUCCESS,NACH,VND0000002,MISSING_REFERENCE,2026-06-28 13:50:03,2026-06
24525,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,TXN0000013106,119714.20,SUCCESS,NETBANKING,VND0000014,VALID,2026-03-26 12:49:42,2026-03
7418,PAYMENT0007582,ACC0017962,BRW0008041,2026-03-26 12:49:42,NaN,119714.20,SUCCESS,NETBANKING,VND0000014,MISSING_REFERENCE,2026-03-26 12:49:42,2026-03



Duplicate payment reference records: 7328


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,quality_flag,event_at_dt,payment_month
3395,PAYMENT0003478,ACC0022277,BRW0011422,2026-06-09 12:22:41,TXN0000000027,43010.39,SUCCESS,NACH,VND0000010,VALID,2026-06-09 12:22:41,2026-06
1363,PAYMENT0001388,ACC0019483,BRW0003324,2026-01-31 05:00:49,TXN0000000027,53610.92,FAILED,CARD,VND0000005,VALID,2026-01-31 05:00:49,2026-01
20764,PAYMENT0021183,ACC0011292,BRW0006242,2026-04-10 12:10:27,TXN0000000032,15714.97,SUCCESS,CASH,VND0000005,VALID,2026-04-10 12:10:27,2026-04
5182,PAYMENT0005307,ACC0003975,BRW0007408,2026-05-24 14:49:16,TXN0000000032,18086.24,PENDING,CARD,VND0000003,VALID,2026-05-24 14:49:16,2026-05
2814,PAYMENT0002885,ACC0021199,BRW0008425,2026-05-03 23:45:49,TXN0000000032,104969.97,SUCCESS,NACH,VND0000007,VALID,2026-05-03 23:45:49,2026-05
2769,PAYMENT0002840,ACC0022601,BRW0006594,2026-02-01 22:47:46,TXN0000000113,1139.66,SUCCESS,CARD,VND0000003,VALID,2026-02-01 22:47:46,2026-02
13014,PAYMENT0013293,ACC0019961,BRW0006917,2026-01-17 20:51:05,TXN0000000113,55554.44,REVERSED,CARD,VND0000007,VALID,2026-01-17 20:51:05,2026-01
18535,PAYMENT0018918,ACC0012876,BRW0010206,2026-08-03 02:17:02,TXN0000000113,101681.86,SUCCESS,CARD,VND0000010,VALID,2026-08-03 02:17:02,2026-08
6522,PAYMENT0006669,ACC0003254,BRW0007989,2026-06-11 02:48:54,TXN0000000132,73506.80,SUCCESS,NACH,VND0000013,VALID,2026-06-11 02:48:54,2026-06
21960,PAYMENT0022397,ACC0012515,BRW0010930,2026-02-25 12:02:00,TXN0000000132,56286.88,SUCCESS,CARD,VND0000005,VALID,2026-02-25 12:02:00,2026-02



Exact duplicate rows: 0


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,quality_flag,event_at_dt,payment_month



Duplicate payment ID summary:


,payment_id,records,accounts,total_amount,statuses
0,PAYMENT0001390,2,1,170410.32,1
1,PAYMENT0002098,2,1,11141.84,1
2,PAYMENT0002303,2,1,118859.72,1
3,PAYMENT0005191,2,1,214470.94,1
4,PAYMENT0007582,2,1,239428.40,1
5,PAYMENT0007751,2,1,53538.68,1
6,PAYMENT0008752,2,1,15829.80,1
7,PAYMENT0010558,2,1,181773.52,1
8,PAYMENT0016862,2,1,251834.08,1
9,PAYMENT0018134,2,1,22488.34,1


In [105]:
# ============================================================
# 88. PAYMENT REFERENCE INVESTIGATION
# ============================================================

reference_analysis = (
    payment_forensics
    .groupby("payment_reference", dropna=False)
    .agg(
        records=("payment_reference", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_statuses=("payment_status", "nunique")
    )
    .reset_index()
)

# References associated with multiple payment events
repeated_reference_analysis = reference_analysis[
    reference_analysis["records"] > 1
].sort_values(
    "records",
    ascending=False
)

print(
    "Repeated payment references:",
    len(repeated_reference_analysis)
)

display(
    repeated_reference_analysis.head(20)
)

Repeated payment references: 3284


,payment_reference,records,unique_payment_ids,unique_accounts,unique_amounts,unique_statuses
20483,NaN,382,382,380,382,4
12866,TXN0000044312,5,5,5,5,2
9574,TXN0000033038,4,4,4,4,3
9445,TXN0000032669,4,4,4,4,1
3809,TXN0000013291,4,4,4,4,2
5800,TXN0000020227,4,4,4,4,1
3392,TXN0000011850,4,4,4,4,2
8077,TXN0000027954,4,4,4,4,2
7336,TXN0000025345,4,4,4,4,2
511,TXN0000001778,4,4,4,4,2


In [106]:
# ============================================================
# 89. DISTINGUISH REPEATED REFERENCES FROM DUPLICATE EVENTS
# ============================================================

reference_event_check = (
    payment_forensics
    .groupby("payment_reference", dropna=False)
    .agg(
        records=("payment_reference", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_timestamps=("event_at", "nunique")
    )
    .reset_index()
)

# References repeated but representing different payment events
reference_reuse = reference_event_check[
    (reference_event_check["records"] > 1) &
    (
        (reference_event_check["unique_payment_ids"] > 1) |
        (reference_event_check["unique_accounts"] > 1) |
        (reference_event_check["unique_amounts"] > 1) |
        (reference_event_check["unique_timestamps"] > 1)
    )
]

# References where every observed attribute is identical
potential_duplicate_events = reference_event_check[
    (reference_event_check["records"] > 1) &
    (reference_event_check["unique_payment_ids"] == 1) &
    (reference_event_check["unique_accounts"] == 1) &
    (reference_event_check["unique_amounts"] == 1) &
    (reference_event_check["unique_timestamps"] == 1)
]

print("Repeated references representing different event attributes:",
      len(reference_reuse))

print("Repeated references with identical event attributes:",
      len(potential_duplicate_events))

display(reference_reuse.head(20))

Repeated references representing different event attributes: 3284
Repeated references with identical event attributes: 0


,payment_reference,records,unique_payment_ids,unique_accounts,unique_amounts,unique_timestamps
4,TXN0000000027,2,2,2,2,2
7,TXN0000000032,3,3,3,3,3
27,TXN0000000113,3,3,3,3,3
34,TXN0000000132,2,2,2,2,2
35,TXN0000000134,2,2,2,2,2
38,TXN0000000154,2,2,2,2,2
49,TXN0000000186,2,2,2,2,2
52,TXN0000000196,2,2,2,2,2
57,TXN0000000212,2,2,2,2,2
70,TXN0000000271,2,2,2,2,2


In [107]:
# ============================================================
# 90. EXACT DUPLICATE FINANCIAL IMPACT
# ============================================================

exact_duplicate_amount = exact_duplicates["amount"].sum()

successful_exact_duplicates = exact_duplicates[
    exact_duplicates["payment_status"].eq("SUCCESS")
]

successful_exact_duplicate_amount = (
    successful_exact_duplicates["amount"].sum()
)

print("Exact duplicate rows:", len(exact_duplicates))
print(
    "Exact duplicate amount:",
    f"₹{exact_duplicate_amount:,.2f}"
)
print(
    "Successful exact duplicate rows:",
    len(successful_exact_duplicates)
)
print(
    "Successful exact duplicate amount:",
    f"₹{successful_exact_duplicate_amount:,.2f}"
)

Exact duplicate rows: 0
Exact duplicate amount: ₹0.00
Successful exact duplicate rows: 0
Successful exact duplicate amount: ₹0.00


In [108]:
# ============================================================
# 91. DUPLICATE PAYMENT INVESTIGATION — CONCLUSION
# ============================================================

duplicate_payment_conclusion = pd.DataFrame([{
    "finding": "Repeated payment references observed",
    "repeated_references": len(repeated_reference_analysis),
    "exact_duplicate_rows": len(exact_duplicates),
    "successful_exact_duplicate_rows": len(successful_exact_duplicates),
    "successful_exact_duplicate_amount": successful_exact_duplicate_amount,
    "decision": "Do not deduplicate payment references; retain payment events and exclude only verified exact duplicate rows."
}])

display(duplicate_payment_conclusion)

,finding,repeated_references,exact_duplicate_rows,successful_exact_duplicate_rows,successful_exact_duplicate_amount,decision
0,Repeated payment references observed,3284,0,0,0.0,Do not deduplicate payment references; retain ...


## 92. TIMEZONE FORENSICS

In [109]:
# ============================================================
# 92. TIMEZONE FORENSICS
# ============================================================

timestamp_columns = []

for column in payments.columns:
    if "event_at" in column.lower() or "timestamp" in column.lower():
        timestamp_columns.append(column)

print("Potential timestamp columns:")
print(timestamp_columns)

for column in timestamp_columns:
    parsed = pd.to_datetime(
        payments[column],
        errors="coerce"
    )

    print(f"\n{column}")
    print("Valid timestamps:", parsed.notna().sum())
    print("Invalid timestamps:", parsed.isna().sum())
    
    # Detect timezone-aware values
    try:
        print("Timezone:", parsed.dt.tz)
    except:
        print("Timezone: mixed/naive")

Potential timestamp columns:
['event_at', 'event_at_dt']

event_at
Valid timestamps: 24528
Invalid timestamps: 0
Timezone: None

event_at_dt
Valid timestamps: 24528
Invalid timestamps: 0
Timezone: None


In [110]:
# ============================================================
# 93. TIMESTAMP FORMAT / TIMEZONE LABEL CHECK
# ============================================================

timestamp_format_check = payments["event_at"].astype(str)

timezone_labels = timestamp_format_check.str.extract(
    r"(Z|[+-]\d{2}:?\d{2}|UTC|IST|Asia/Kolkata|Asia/Dubai)",
    expand=False
)

print("Timezone-labelled timestamps:", timezone_labels.notna().sum())
print("Timezone-naive timestamps:", timezone_labels.isna().sum())

print("\nSample event_at values:")
display(
    payments["event_at"]
    .drop_duplicates()
    .head(10)
)

Timezone-labelled timestamps: 0
Timezone-naive timestamps: 24528

Sample event_at values:


0    2026-02-27 01:28:12
1    2026-07-23 20:25:20
2    2026-01-11 21:27:46
3    2026-06-16 02:35:21
4    2026-03-03 06:08:23
5    2026-05-25 10:59:51
6    2026-05-08 11:45:34
7    2026-01-16 08:37:42
8    2026-03-31 08:11:38
9    2026-07-20 20:51:33
Name: event_at, dtype: str

In [111]:
# ============================================================
# 94. TIMEZONE SENSITIVITY CHECK
# ============================================================

timestamp_sensitivity = payments.copy()

timestamp_sensitivity["event_at_dt"] = pd.to_datetime(
    timestamp_sensitivity["event_at"],
    errors="coerce"
)

# Compare dates under the source naive timestamp
timestamp_sensitivity["source_date"] = (
    timestamp_sensitivity["event_at_dt"].dt.date
)

# Flag events near midnight where timezone conversion could
# potentially move the event to a different calendar date.
timestamp_sensitivity["hour"] = (
    timestamp_sensitivity["event_at_dt"].dt.hour
)

near_midnight = timestamp_sensitivity[
    (timestamp_sensitivity["hour"] <= 2) |
    (timestamp_sensitivity["hour"] >= 22)
]

print("Total payment records:", len(timestamp_sensitivity))
print(
    "Events between 22:00 and 02:59:",
    len(near_midnight)
)
print(
    "Percentage potentially sensitive to calendar-date shifts:",
    round(len(near_midnight) / len(timestamp_sensitivity) * 100, 2),
    "%"
)

display(
    near_midnight[
        ["payment_id", "event_at", "payment_status", "amount"]
    ].head(20)
)

Total payment records: 24528
Events between 22:00 and 02:59: 5213
Percentage potentially sensitive to calendar-date shifts: 21.25 %


,payment_id,event_at,payment_status,amount
0,PAYMENT0000001,2026-02-27 01:28:12,FAILED,22433.23
3,PAYMENT0000004,2026-06-16 02:35:21,REVERSED,145082.17
12,PAYMENT0000013,2026-05-06 22:30:42,SUCCESS,41543.11
18,PAYMENT0000019,2026-02-19 22:42:24,SUCCESS,102897.37
21,PAYMENT0000022,2026-07-04 23:55:34,PENDING,46845.47
25,PAYMENT0000026,2026-02-07 23:29:31,PENDING,6660.28
27,PAYMENT0000028,2026-05-24 23:43:17,REVERSED,74205.30
40,PAYMENT0000041,2026-04-28 22:03:24,SUCCESS,94558.06
41,PAYMENT0000042,2026-06-11 02:06:40,SUCCESS,18096.51
47,PAYMENT0000048,2026-06-17 01:15:38,SUCCESS,142833.99


In [112]:
# ============================================================
# 95. TIMEZONE FORENSICS — CONCLUSION
# ============================================================

timezone_conclusion = pd.DataFrame([{
    "finding": "Payment timestamps are timezone-naive",
    "total_records": len(timestamp_sensitivity),
    "timezone_labelled_records": int(timezone_labels.notna().sum()),
    "near_midnight_records": len(near_midnight),
    "near_midnight_pct": round(
        len(near_midnight) / len(timestamp_sensitivity) * 100,
        2
    ),
    "decision": (
        "Preserve source timestamps and do not apply an assumed timezone "
        "conversion. Calendar-date analysis should use the supplied "
        "timestamp consistently and document the timezone limitation."
    )
}])

display(timezone_conclusion)

,finding,total_records,timezone_labelled_records,near_midnight_records,near_midnight_pct,decision
0,Payment timestamps are timezone-naive,24528,0,5213,21.25,Preserve source timestamps and do not apply an...


## 96. VENDOR MAPPING FORENSICS

In [113]:
# ============================================================
# 96. VENDOR MAPPING FORENSICS
# ============================================================

vendor_data = vendor_telephony_analysis.copy()

print("Vendor telephony records:", len(vendor_data))
print("Columns:")
print(vendor_data.columns.tolist())

# Check whether vendor identifiers map to multiple attributes
vendor_mapping_check = (
    vendor_data
    .groupby("vendor_id", dropna=False)
    .agg(
        records=("vendor_id", "size"),
        unique_vendor_names=("vendor_name", "nunique")
        if "vendor_name" in vendor_data.columns
        else ("vendor_id", "nunique"),
        unique_phone_numbers=("phone_number", "nunique")
        if "phone_number" in vendor_data.columns
        else ("vendor_id", "nunique")
    )
    .reset_index()
)

display(vendor_mapping_check)

Vendor telephony records: 15
Columns:
['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']


,vendor_id,records,unique_vendor_names,unique_phone_numbers
0,VND0000001,1,1,1
1,VND0000002,1,1,1
2,VND0000003,1,1,1
3,VND0000004,1,1,1
4,VND0000005,1,1,1
5,VND0000006,1,1,1
6,VND0000007,1,1,1
7,VND0000008,1,1,1
8,VND0000009,1,1,1
9,VND0000010,1,1,1


In [114]:
# ============================================================
# 96. VENDOR MAPPING FORENSICS
# ============================================================

vendor_data = vendor_telephony_analysis.copy()

vendor_mapping_check = (
    vendor_data
    .groupby("vendor_id", dropna=False)
    .agg(
        records=("vendor_id", "size"),
        unique_vendor_names=("vendor_name", "nunique"),
        unique_vendor_accounts=("vendor_account_id", "nunique"),
        unique_timezones=("timezone", "nunique"),
        unique_statuses=("status", "nunique"),
        unique_schema_versions=("schema_version", "nunique")
    )
    .reset_index()
)

print("Vendor IDs:", vendor_mapping_check["vendor_id"].nunique())
print(
    "Vendor IDs with multiple vendor names:",
    (vendor_mapping_check["unique_vendor_names"] > 1).sum()
)
print(
    "Vendor IDs with multiple vendor accounts:",
    (vendor_mapping_check["unique_vendor_accounts"] > 1).sum()
)
print(
    "Vendor IDs with multiple timezones:",
    (vendor_mapping_check["unique_timezones"] > 1).sum()
)
print(
    "Vendor IDs with multiple schema versions:",
    (vendor_mapping_check["unique_schema_versions"] > 1).sum()
)

display(vendor_mapping_check)

Vendor IDs: 15
Vendor IDs with multiple vendor names: 0
Vendor IDs with multiple vendor accounts: 0
Vendor IDs with multiple timezones: 0
Vendor IDs with multiple schema versions: 0


,vendor_id,records,unique_vendor_names,unique_vendor_accounts,unique_timezones,unique_statuses,unique_schema_versions
0,VND0000001,1,1,1,1,1,1
1,VND0000002,1,1,1,1,1,1
2,VND0000003,1,1,1,1,1,1
3,VND0000004,1,1,1,1,1,1
4,VND0000005,1,1,1,1,1,1
5,VND0000006,1,1,1,1,1,1
6,VND0000007,1,1,1,1,1,1
7,VND0000008,1,1,1,1,1,1
8,VND0000009,1,1,1,1,1,1
9,VND0000010,1,1,1,1,1,1


In [115]:
# ============================================================
# 97. VENDOR MAPPING FORENSICS — CONCLUSION
# ============================================================

vendor_conclusion = pd.DataFrame([{
    "finding": "Vendor mapping is stable",
    "vendor_ids": vendor_mapping_check["vendor_id"].nunique(),
    "multiple_vendor_names": int(
        (vendor_mapping_check["unique_vendor_names"] > 1).sum()
    ),
    "multiple_vendor_accounts": int(
        (vendor_mapping_check["unique_vendor_accounts"] > 1).sum()
    ),
    "multiple_timezones": int(
        (vendor_mapping_check["unique_timezones"] > 1).sum()
    ),
    "multiple_schema_versions": int(
        (vendor_mapping_check["unique_schema_versions"] > 1).sum()
    ),
    "decision": (
        "No vendor mapping change was detected. "
        "Retain vendor_id as the stable vendor identifier."
    )
}])

display(vendor_conclusion)

,finding,vendor_ids,multiple_vendor_names,multiple_vendor_accounts,multiple_timezones,multiple_schema_versions,decision
0,Vendor mapping is stable,15,0,0,0,0,No vendor mapping change was detected. Retain ...


## 98. AGENT IDENTITY FORENSICS

In [116]:
# ============================================================
# 98. AGENT IDENTITY FORENSICS
# ============================================================

agent_data = agents_analysis.copy()

print("Agent records:", len(agent_data))
print("Agent columns:")
print(agent_data.columns.tolist())

agent_identity_check = (
    agent_data
    .groupby("agent_id", dropna=False)
    .agg(
        records=("agent_id", "size"),
        unique_names=("agent_name", "nunique")
        if "agent_name" in agent_data.columns
        else ("agent_id", "nunique"),
        unique_vendors=("vendor_id", "nunique")
        if "vendor_id" in agent_data.columns
        else ("agent_id", "nunique"),
        unique_statuses=("status", "nunique")
        if "status" in agent_data.columns
        else ("agent_id", "nunique")
    )
    .reset_index()
)

print(
    "Agent IDs with multiple names:",
    (agent_identity_check["unique_names"] > 1).sum()
)

print(
    "Agent IDs with multiple vendors:",
    (agent_identity_check["unique_vendors"] > 1).sum()
)

print(
    "Agent IDs with multiple statuses:",
    (agent_identity_check["unique_statuses"] > 1).sum()
)

display(agent_identity_check)

Agent records: 30000
Agent columns:
['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']
Agent IDs with multiple names: 1000
Agent IDs with multiple vendors: 1000
Agent IDs with multiple statuses: 1000


,agent_id,records,unique_names,unique_vendors,unique_statuses
0,AGT0000001,23,10,10,3
1,AGT0000002,23,10,12,3
2,AGT0000003,28,10,12,3
3,AGT0000004,28,10,12,3
4,AGT0000005,29,10,15,3
...,...,...,...,...,...
995,AGT0000996,26,10,12,3
996,AGT0000997,42,10,15,3
997,AGT0000998,32,10,13,3
998,AGT0000999,27,8,11,3


In [117]:
# ============================================================
# 99. AGENT IDENTITY CHANGE EXAMPLES
# ============================================================

agent_identity_issues = agent_identity_check[
    (agent_identity_check["unique_names"] > 1) |
    (agent_identity_check["unique_vendors"] > 1) |
    (agent_identity_check["unique_statuses"] > 1)
]

print(
    "Agent IDs requiring identity investigation:",
    len(agent_identity_issues)
)

example_agent_ids = agent_identity_issues["agent_id"].head(10).tolist()

display(
    agent_data[
        agent_data["agent_id"].isin(example_agent_ids)
    ].sort_values(
        ["agent_id", "updated_at"]
    )
)

Agent IDs requiring identity investigation: 1000


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
9554,AGT0000001,EMP00883,Sneha Das,VND0000010,T3,INACTIVE,2025-09-15 12:18:43,2025-02-08 10:18:20
3016,AGT0000001,EMP00285,Sneha Das,VND0000013,DIGITAL,INACTIVE,2025-05-02 11:13:31,2025-02-14 12:38:08
18301,AGT0000001,EMP00191,Priya Mehta,VND0000015,FIELD,ACTIVE,2025-11-04 07:01:56,2025-03-09 16:24:54
21572,AGT0000001,EMP00745,Vikram Shah,VND0000008,T3,ACTIVE,2024-02-08 16:05:41,2025-03-09 22:26:39
8967,AGT0000001,EMP01082,Neha Singh,VND0000013,T1,INACTIVE,2025-05-02 01:36:54,2025-04-25 02:56:44
...,...,...,...,...,...,...,...,...
21475,AGT0000010,EMP00522,Priya Mehta,VND0000005,T3,SUSPENDED,2024-07-06 11:28:41,2026-02-02 17:03:34
11766,AGT0000010,EMP00571,Neha Singh,VND0000001,T1,ACTIVE,2024-11-26 17:05:44,2026-02-12 12:48:14
8014,AGT0000010,EMP00688,Amit Kumar,VND0000008,T3,ACTIVE,2024-06-10 21:27:27,2026-03-25 06:38:53
11452,AGT0000010,EMP01063,Amit Kumar,VND0000012,FIELD,SUSPENDED,2025-07-24 05:38:59,2026-05-21 16:12:39


In [118]:
# ============================================================
# 99. AGENT IDENTITY CHANGE EXAMPLES
# ============================================================

example_agent_ids = agent_identity_issues["agent_id"].head(3).tolist()

display(
    agent_data[
        agent_data["agent_id"].isin(example_agent_ids)
    ][
        [
            "agent_id",
            "employee_code",
            "agent_name",
            "vendor_id",
            "team",
            "status",
            "joined_at",
            "updated_at"
        ]
    ].sort_values(["agent_id", "updated_at"])
)

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
9554,AGT0000001,EMP00883,Sneha Das,VND0000010,T3,INACTIVE,2025-09-15 12:18:43,2025-02-08 10:18:20
3016,AGT0000001,EMP00285,Sneha Das,VND0000013,DIGITAL,INACTIVE,2025-05-02 11:13:31,2025-02-14 12:38:08
18301,AGT0000001,EMP00191,Priya Mehta,VND0000015,FIELD,ACTIVE,2025-11-04 07:01:56,2025-03-09 16:24:54
21572,AGT0000001,EMP00745,Vikram Shah,VND0000008,T3,ACTIVE,2024-02-08 16:05:41,2025-03-09 22:26:39
8967,AGT0000001,EMP01082,Neha Singh,VND0000013,T1,INACTIVE,2025-05-02 01:36:54,2025-04-25 02:56:44
...,...,...,...,...,...,...,...,...
16909,AGT0000003,EMP00184,Rahul Verma,VND0000002,DIGITAL,ACTIVE,2024-09-16 06:00:18,2026-02-18 22:20:51
4169,AGT0000003,EMP00534,Ananya Rao,VND0000008,T1,INACTIVE,2024-04-06 14:13:08,2026-03-07 12:23:56
851,AGT0000003,EMP00655,Rohan Patel,VND0000014,FIELD,ACTIVE,2025-07-09 14:05:09,2026-05-05 01:40:44
4354,AGT0000003,EMP00717,Sneha Das,VND0000012,FIELD,ACTIVE,2025-02-01 09:03:57,2026-07-14 07:52:14


In [119]:
# ============================================================
# 100. AGENT IDENTITY IMPACT
# ============================================================

agent_identity_impact = pd.DataFrame([{
    "total_agent_records": len(agent_data),
    "unique_agent_ids": agent_data["agent_id"].nunique(),
    "agent_ids_with_multiple_names": (
        agent_identity_check["unique_names"] > 1
    ).sum(),
    "agent_ids_with_multiple_employee_codes": (
        agent_data.groupby("agent_id")["employee_code"].nunique() > 1
    ).sum(),
    "agent_ids_with_multiple_vendors": (
        agent_identity_check["unique_vendors"] > 1
    ).sum(),
    "agent_ids_with_multiple_statuses": (
        agent_identity_check["unique_statuses"] > 1
    ).sum()
}])

display(agent_identity_impact)

,total_agent_records,unique_agent_ids,agent_ids_with_multiple_names,agent_ids_with_multiple_employee_codes,agent_ids_with_multiple_vendors,agent_ids_with_multiple_statuses
0,30000,1000,1000,1000,1000,1000


In [120]:
# ============================================================
# 101. AGENT IDENTITY FORENSICS — CONCLUSION
# ============================================================

agent_identity_conclusion = pd.DataFrame([{
    "finding": "Agent identifiers are reused across conflicting identities",
    "agent_records": len(agent_data),
    "unique_agent_ids": agent_data["agent_id"].nunique(),
    "affected_agent_ids": len(agent_identity_issues),
    "decision": (
        "Do not treat agent_id as a unique person-level key. "
        "Retain the records and flag identity conflicts. "
        "Use employee_code and supporting attributes for identity "
        "resolution where validated."
    )
}])

display(agent_identity_conclusion)

,finding,agent_records,unique_agent_ids,affected_agent_ids,decision
0,Agent identifiers are reused across conflictin...,30000,1000,1000,Do not treat agent_id as a unique person-level...


## 102. DENOMINATOR FORENSICS

In [121]:
# ============================================================
# 102. DENOMINATOR FORENSICS
# ============================================================

denominator_checks = pd.DataFrame([
    {
        "denominator": "Total outstanding portfolio",
        "value": total_outstanding,
        "recovery_rate_pct": (
            recovered_amount / total_outstanding * 100
        )
    },
    {
        "denominator": "Accounts with successful payment",
        "value": successful_payment_accounts,
        "recovery_rate_pct": np.nan
    },
    {
        "denominator": "Total accounts",
        "value": total_accounts,
        "recovery_rate_pct": np.nan
    }
])

print("Reported recovery rate:", reported_recovery_rate, "%")
print("Calculated recovery rate:", round(recovery_rate, 2), "%")
print(
    "Difference:",
    round(difference_percentage_points, 2),
    "percentage points"
)

display(denominator_checks)

Reported recovery rate: 11.0 %
Calculated recovery rate: 12.31 %
Difference: 1.31 percentage points


,denominator,value,recovery_rate_pct
0,Total outstanding portfolio,1.048904e+10,12.312505
1,Accounts with successful payment,1.310900e+04,NaN
2,Total accounts,3.000000e+04,NaN


In [122]:
# ============================================================
# 103. DENOMINATOR FORENSICS — CONCLUSION
# ============================================================

denominator_conclusion = pd.DataFrame([{
    "finding": "Reported 11% recovery rate does not reconcile with core denominator",
    "reported_rate_pct": reported_recovery_rate,
    "calculated_rate_pct": recovery_rate,
    "difference_pp": difference_percentage_points,
    "denominator": "Total outstanding amount in accounts",
    "decision": (
        "Use total outstanding amount as the documented denominator. "
        "Do not alter the denominator to force reconciliation with the "
        "reported 11% figure."
    )
}])

display(denominator_conclusion)

,finding,reported_rate_pct,calculated_rate_pct,difference_pp,denominator,decision
0,Reported 11% recovery rate does not reconcile ...,11.0,12.312505,1.312505,Total outstanding amount in accounts,Use total outstanding amount as the documented...


## 104. ATTRIBUTION FORENSICS

In [123]:
# ============================================================
# 104. ATTRIBUTION FORENSICS
# ============================================================

print("Attribution records:", len(attribution_base))

attribution_check = attribution_base.copy()

# Check whether one payment is attributed to multiple interactions
payment_attribution_check = (
    attribution_check
    .groupby("payment_id", dropna=False)
    .agg(
        attribution_records=("payment_id", "size"),
        unique_accounts=("account_id", "nunique"),
        unique_campaigns=("campaign_id", "nunique")
        if "campaign_id" in attribution_check.columns
        else ("payment_id", "nunique"),
        unique_agents=("agent_id", "nunique")
        if "agent_id" in attribution_check.columns
        else ("payment_id", "nunique"),
        unique_vendors=("vendor_id", "nunique")
        if "vendor_id" in attribution_check.columns
        else ("payment_id", "nunique")
    )
    .reset_index()
)

multiple_attribution = payment_attribution_check[
    payment_attribution_check["attribution_records"] > 1
]

print(
    "Payments with multiple attribution records:",
    len(multiple_attribution)
)

print(
    "Total attribution records:",
    len(attribution_check)
)

display(multiple_attribution.head(20))

Attribution records: 1608
Payments with multiple attribution records: 25
Total attribution records: 1608


,payment_id,attribution_records,unique_accounts,unique_campaigns,unique_agents,unique_vendors
8,PAYMENT0000149,2,1,1,1,1
54,PAYMENT0000850,2,1,1,1,1
138,PAYMENT0002299,2,1,1,1,1
145,PAYMENT0002478,2,1,1,1,1
306,PAYMENT0004999,2,1,1,1,1
510,PAYMENT0007752,2,1,1,1,1
541,PAYMENT0008256,2,1,1,1,1
611,PAYMENT0009343,2,1,1,1,1
635,PAYMENT0009839,2,1,1,1,1
671,PAYMENT0010607,2,1,1,1,1


In [124]:
# ============================================================
# 105. ATTRIBUTION DOUBLE-COUNTING CHECK
# ============================================================

multi_attribution_ids = multiple_attribution["payment_id"]

multi_attribution_detail = attribution_check[
    attribution_check["payment_id"].isin(multi_attribution_ids)
].copy()

attribution_double_count_check = (
    multi_attribution_detail
    .groupby("payment_id")
    .agg(
        attribution_records=("payment_id", "size"),
        attributed_amount=("amount", "sum"),
        unique_accounts=("account_id", "nunique")
    )
    .reset_index()
)

print(
    "Payments with multiple attribution records:",
    len(attribution_double_count_check)
)

print(
    "Attributed amount across multi-attributed payments:",
    f"₹{attribution_double_count_check['attributed_amount'].sum():,.2f}"
)

display(attribution_double_count_check)

Payments with multiple attribution records: 25
Attributed amount across multi-attributed payments: ₹3,564,617.74


,payment_id,attribution_records,attributed_amount,unique_accounts
0,PAYMENT0000149,2,193532.24,1
1,PAYMENT0000850,2,101492.16,1
2,PAYMENT0002299,2,64104.34,1
3,PAYMENT0002478,2,247641.48,1
4,PAYMENT0004999,2,10508.24,1
5,PAYMENT0007752,2,112198.26,1
6,PAYMENT0008256,2,225215.02,1
7,PAYMENT0009343,2,85598.08,1
8,PAYMENT0009839,2,68667.00,1
9,PAYMENT0010607,2,158790.86,1


In [125]:
# ============================================================
# 106. ATTRIBUTION AMOUNT DUPLICATION CHECK
# ============================================================

payment_amount_check = (
    multi_attribution_detail
    .groupby("payment_id")
    .agg(
        attribution_records=("payment_id", "size"),
        unique_amounts=("amount", "nunique"),
        payment_amount=("amount", "first"),
        attributed_amount=("amount", "sum")
    )
    .reset_index()
)

payment_amount_check["excess_attributed_amount"] = (
    payment_amount_check["attributed_amount"]
    - payment_amount_check["payment_amount"]
)

print(
    "Payments where attribution creates excess amount:",
    (
        payment_amount_check["excess_attributed_amount"] > 0
    ).sum()
)

print(
    "Potential excess attributed amount:",
    f"₹{payment_amount_check['excess_attributed_amount'].sum():,.2f}"
)

display(payment_amount_check)

Payments where attribution creates excess amount: 25
Potential excess attributed amount: ₹1,782,308.87


,payment_id,attribution_records,unique_amounts,payment_amount,attributed_amount,excess_attributed_amount
0,PAYMENT0000149,2,1,96766.12,193532.24,96766.12
1,PAYMENT0000850,2,1,50746.08,101492.16,50746.08
2,PAYMENT0002299,2,1,32052.17,64104.34,32052.17
3,PAYMENT0002478,2,1,123820.74,247641.48,123820.74
4,PAYMENT0004999,2,1,5254.12,10508.24,5254.12
5,PAYMENT0007752,2,1,56099.13,112198.26,56099.13
6,PAYMENT0008256,2,1,112607.51,225215.02,112607.51
7,PAYMENT0009343,2,1,42799.04,85598.08,42799.04
8,PAYMENT0009839,2,1,34333.50,68667.00,34333.50
9,PAYMENT0010607,2,1,79395.43,158790.86,79395.43


In [127]:
# ============================================================
# 107. ATTRIBUTION FORENSICS — CONCLUSION
# ============================================================

attribution_conclusion = pd.DataFrame([{
    "finding": "Multiple collection touches can receive attribution for the same payment",
    "multi_attributed_payments": len(payment_amount_check),
    "potential_excess_attributed_amount": (
        payment_amount_check["excess_attributed_amount"].sum()
    ),
    "decision": (
        "Do not use raw attributed recovery as a replacement for total "
        "successful recovery. Preserve the payment-level recovery metric "
        "as the source of truth and treat campaign, agent, and vendor "
        "attribution as an analytical view. Apply a mutually exclusive "
        "attribution rule before comparing attributed recovery across "
        "operational dimensions."
    )
}])

display(attribution_conclusion)

,finding,multi_attributed_payments,potential_excess_attributed_amount,decision
0,Multiple collection touches can receive attrib...,25,1782308.87,Do not use raw attributed recovery as a replac...


## 108. STATISTICAL INVESTIGATION

In [129]:
# ============================================================
# 108. STATISTICAL INVESTIGATION — RECOVERY BY RISK SEGMENT
# ============================================================

from scipy.stats import chi2_contingency

# Account-level payment outcome by risk segment
risk_payment_outcome = accounts[
    ["account_id", "risk_segment"]
].copy()

paying_account_ids = set(
    successful_payments["account_id"].dropna()
)

risk_payment_outcome["successful_payment"] = (
    risk_payment_outcome["account_id"]
    .isin(paying_account_ids)
)

contingency_table = pd.crosstab(
    risk_payment_outcome["risk_segment"],
    risk_payment_outcome["successful_payment"]
)

chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", round(chi2, 4))
print("p-value:", round(p_value, 6))
print("Degrees of freedom:", dof)

display(contingency_table)

if p_value < 0.05:
    print("Result: Payment outcome differs significantly across risk segments.")
else:
    print("Result: No statistically significant difference detected.")

Chi-square statistic: 2.538
p-value: 0.468456
Degrees of freedom: 3


successful_payment,False,True
risk_segment,,
HIGH,4286,3266
LOW,4173,3340
MEDIUM,4250,3283
NPA,4182,3220


Result: No statistically significant difference detected.


In [130]:
# ============================================================
# 109. RISK SEGMENT STATISTICAL TEST — CONCLUSION
# ============================================================

risk_statistical_conclusion = pd.DataFrame([{
    "analysis": "Successful payment outcome by risk segment",
    "chi_square": chi2,
    "p_value": p_value,
    "degrees_of_freedom": dof,
    "significance_level": 0.05,
    "result": "Not statistically significant",
    "decision": (
        "Do not conclude that risk segment independently drives "
        "successful payment outcome based on this test."
    )
}])

display(risk_statistical_conclusion)

,analysis,chi_square,p_value,degrees_of_freedom,significance_level,result,decision
0,Successful payment outcome by risk segment,2.538039,0.468456,3,0.05,Not statistically significant,Do not conclude that risk segment independentl...


## 110. STATISTICAL INVESTIGATION — RECOVERY BY DPD

In [131]:
# ============================================================
# 110. STATISTICAL INVESTIGATION — RECOVERY BY DPD
# ============================================================

dpd_payment_outcome = accounts[
    ["account_id", "dpd_bucket"]
].copy()

dpd_payment_outcome["successful_payment"] = (
    dpd_payment_outcome["account_id"]
    .isin(paying_account_ids)
)

dpd_contingency_table = pd.crosstab(
    dpd_payment_outcome["dpd_bucket"],
    dpd_payment_outcome["successful_payment"]
)

dpd_chi2, dpd_p_value, dpd_dof, dpd_expected = chi2_contingency(
    dpd_contingency_table
)

print("Chi-square statistic:", round(dpd_chi2, 4))
print("p-value:", round(dpd_p_value, 6))
print("Degrees of freedom:", dpd_dof)

display(dpd_contingency_table)

if dpd_p_value < 0.05:
    print("Result: Payment outcome differs significantly across DPD buckets.")
else:
    print("Result: No statistically significant difference detected.")

Chi-square statistic: 8.0098
p-value: 0.091219
Degrees of freedom: 4


successful_payment,False,True
dpd_bucket,,
0-30,7710,5855
120+,1548,1146
31-60,3026,2488
61-90,3055,2413
91-120,1552,1207


Result: No statistically significant difference detected.


In [132]:
# ============================================================
# 111. DPD STATISTICAL TEST — CONCLUSION
# ============================================================

dpd_statistical_conclusion = pd.DataFrame([{
    "analysis": "Successful payment outcome by DPD bucket",
    "chi_square": dpd_chi2,
    "p_value": dpd_p_value,
    "degrees_of_freedom": dpd_dof,
    "significance_level": 0.05,
    "result": "Not statistically significant",
    "decision": (
        "Do not conclude that DPD bucket independently drives "
        "successful payment outcome based on this test."
    )
}])

display(dpd_statistical_conclusion)

,analysis,chi_square,p_value,degrees_of_freedom,significance_level,result,decision
0,Successful payment outcome by DPD bucket,8.009836,0.091219,4,0.05,Not statistically significant,Do not conclude that DPD bucket independently ...


## 112. STATISTICAL INVESTIGATION — PAYMENT OUTCOME BY CHANNEL

In [134]:
# ============================================================
# 112. STATISTICAL INVESTIGATION — PAYMENT OUTCOME BY CHANNEL
# ============================================================

channel_outcome = (
    campaign_performance[
        ["channel", "targeted_accounts", "paying_accounts"]
    ]
    .groupby("channel", dropna=False)
    .agg(
        targeted_accounts=("targeted_accounts", "sum"),
        paying_accounts=("paying_accounts", "sum")
    )
    .reset_index()
)

channel_outcome["non_paying_accounts"] = (
    channel_outcome["targeted_accounts"]
    - channel_outcome["paying_accounts"]
)

channel_contingency_table = channel_outcome[
    [
        "paying_accounts",
        "non_paying_accounts"
    ]
].set_index(channel_outcome["channel"])

channel_chi2, channel_p_value, channel_dof, channel_expected = (
    chi2_contingency(channel_contingency_table)
)

print("Chi-square statistic:", round(channel_chi2, 4))
print("p-value:", round(channel_p_value, 6))
print("Degrees of freedom:", channel_dof)

display(channel_contingency_table)

if channel_p_value < 0.05:
    print("Result: Payment outcome differs significantly across channels.")
else:
    print("Result: No statistically significant difference detected.")

Chi-square statistic: 0.9687
p-value: 0.914505
Degrees of freedom: 4


,paying_accounts,non_paying_accounts
channel,,
FIELD,241,6495
MIXED,308,8388
SMS,351,10125
VOICE,258,7197
WHATSAPP,403,10949


Result: No statistically significant difference detected.


In [135]:
channel_chi2, channel_p_value, channel_dof, channel_expected = (
    chi2_contingency(channel_contingency_table)
)

## 113. COUNTERFACTUAL METHODOLOGY

The analysis is observational and does not establish causal impact.

A counterfactual evaluation should compare collection outcomes for treated
accounts against a comparable untreated/control population.

For future campaign evaluation:

1. Define the treatment as exposure to a specific campaign/channel.
2. Define an eligible control population that was not exposed during the
   evaluation window.
3. Match or stratify treatment and control accounts using pre-treatment
   characteristics such as DPD bucket, risk segment, outstanding amount,
   loan type, and prior collection activity.
4. Measure successful payment rate and recovered amount over the same
   post-treatment window.
5. Estimate incremental recovery as the difference between treatment and
   comparable control outcomes.
6. Report confidence intervals and statistical significance.
7. Avoid interpreting raw campaign recovery differences as causal effects
   when treatment groups have materially different portfolio composition.

The current dataset supports observational campaign analysis, but a validated
causal estimate requires a clearly defined untreated/control population and
pre-treatment comparability.

## 114. PRODUCTION ANALYTICS DESIGN

### Grain and Primary Keys

Each analytical table retains its native business grain.

- `accounts`: one row per account; primary key = `account_id`
- `payments`: one row per payment event; primary key = `payment_id`
- `calls`: one row per call event; primary key = `call_id`
- `promises_to_pay`: one row per PTP event
- Event tables retain their event-level identifiers where available.

Repeated identifiers are treated as data-quality conditions and are not
automatically removed without evidence of duplication.

### Data Lineage

```text
Raw source tables
      ↓
Profiling & validation
      ↓
Golden datasets / quality flags
      ↓
Analytical tables
      ↓
SQL + Python metrics
      ↓
Power BI semantic model
      ↓
Executive dashboard

## 115. FINAL FORENSIC FINDINGS

The forensic investigation identified the following additional findings:

- Repeated payment references were observed, but no exact duplicate payment
  rows were identified. Payment references are therefore retained rather than
  deduplicated automatically.

- Payment timestamps are timezone-naive. No timezone conversion is applied
  because the source timezone is not explicitly provided. 21.25% of payment
  events occur near midnight and may therefore be sensitive to calendar-date
  shifts under an assumed timezone conversion.

- Vendor mapping is stable across the available vendor records. No conflicting
  vendor names, vendor accounts, timezones, or schema versions were detected
  for the same vendor identifier.

- Agent identifiers are not reliable unique person-level identifiers.
  1,000 agent IDs show conflicting names, employee codes, vendors, and/or
  statuses. Agent-level comparisons therefore require identity-resolution
  controls.

- The reported 11% recovery rate does not reconcile with the calculated
  12.31% recovery rate using total outstanding amount as the denominator.
  The denominator should not be altered to force reconciliation.

- Attribution analysis identified 25 payments with multiple attribution
  records and a potential excess attributed amount of ₹1,782,308.87.
  Attributed recovery should therefore not replace the source-of-truth
  successful-payment recovery metric.

These findings should be considered when interpreting operational,
campaign, agent, vendor, and management-level recovery comparisons.

In [136]:
# ============================================================
# 116. FINAL FINDINGS — UPDATED FORENSIC RESULTS
# ============================================================

final_findings_updated = pd.DataFrame([
    {
        "finding_id": "F-01",
        "area": "Recovery",
        "finding": "Calculated recovery rate vs reported 11%",
        "evidence": "Calculated recovery rate = 12.31%; reported rate = 11.00%",
        "business_impact": "The reported recovery figure does not reconcile with the core denominator.",
        "recommendation": "Use the reconciled recovery definition consistently in management reporting."
    },
    {
        "finding_id": "F-02",
        "area": "Portfolio Mix",
        "finding": "Recovery varies descriptively across DPD and risk segments",
        "evidence": "DPD/risk mix analysis; risk p-value = 0.468456; DPD p-value = 0.091219.",
        "business_impact": "Portfolio composition can influence aggregate recovery, but statistical tests do not establish independent effects.",
        "recommendation": "Track recovery using consistent DPD and risk-segment cohorts."
    },
    {
        "finding_id": "F-03",
        "area": "Campaigns",
        "finding": "Campaign and channel performance varies",
        "evidence": "campaign_performance.csv and channel_performance.csv.",
        "business_impact": "Different collection strategies may be associated with different recovery outcomes.",
        "recommendation": "Compare campaigns using common targeting and attribution definitions before scaling a strategy."
    },
    {
        "finding_id": "F-04",
        "area": "Operations",
        "finding": "Agent and vendor performance varies",
        "evidence": "agent_performance.csv and vendor_performance.csv; 1,000 agent IDs have identity conflicts.",
        "business_impact": "Operational comparisons may be affected by unstable agent identifiers.",
        "recommendation": "Resolve agent identity before using agent-level metrics for management decisions."
    },
    {
        "finding_id": "F-05",
        "area": "PTP",
        "finding": "Recorded PTPs should be validated against subsequent successful payments",
        "evidence": "Observed PTP fulfillment rate = 7.09%.",
        "business_impact": "PTP volume alone may overstate actual collection effectiveness.",
        "recommendation": "Track PTP fulfillment using independently matched payment outcomes."
    },
    {
        "finding_id": "F-06",
        "area": "Data Quality",
        "finding": "Data-quality and attribution limitations affect interpretation",
        "evidence": "Payment, timestamp, agent identity, and attribution forensics.",
        "business_impact": "Unresolved data issues can distort recovery and operational comparisons.",
        "recommendation": "Maintain explicit data-quality flags and metric definitions in recurring reporting."
    },
    {
        "finding_id": "F-07",
        "area": "Payments",
        "finding": "Repeated payment references are not confirmed duplicate events",
        "evidence": "3,284 repeated references; 0 exact duplicate rows; ₹0 exact-duplicate financial impact.",
        "business_impact": "Automatic reference-based deduplication could incorrectly remove legitimate payment events.",
        "recommendation": "Do not deduplicate payment references without validated business-key evidence."
    },
    {
        "finding_id": "F-08",
        "area": "Timestamps",
        "finding": "Payment timestamps are timezone-naive",
        "evidence": "0 timezone-labelled records; 5,213 events (21.25%) occur near midnight.",
        "business_impact": "Calendar-date analysis may be sensitive to an assumed timezone conversion.",
        "recommendation": "Preserve source timestamps and document the timezone limitation."
    },
    {
        "finding_id": "F-09",
        "area": "Attribution",
        "finding": "Multiple attribution records can double-count recovery",
        "evidence": "25 payments with multiple attribution records; ₹1,782,308.87 potential excess attributed recovery.",
        "business_impact": "Raw attributed recovery can overstate campaign, agent, or vendor contribution.",
        "recommendation": "Use successful payments as the source of truth and apply mutually exclusive attribution."
    }
])

final_findings_updated.to_csv(
    PROCESSED_DIR / "final_findings.csv",
    index=False
)

display(final_findings_updated)

,finding_id,area,finding,evidence,business_impact,recommendation
0,F-01,Recovery,Calculated recovery rate vs reported 11%,Calculated recovery rate = 12.31%; reported ra...,The reported recovery figure does not reconcil...,Use the reconciled recovery definition consist...
1,F-02,Portfolio Mix,Recovery varies descriptively across DPD and r...,DPD/risk mix analysis; risk p-value = 0.468456...,Portfolio composition can influence aggregate ...,Track recovery using consistent DPD and risk-s...
2,F-03,Campaigns,Campaign and channel performance varies,campaign_performance.csv and channel_performan...,Different collection strategies may be associa...,Compare campaigns using common targeting and a...
3,F-04,Operations,Agent and vendor performance varies,agent_performance.csv and vendor_performance.c...,Operational comparisons may be affected by uns...,Resolve agent identity before using agent-leve...
4,F-05,PTP,Recorded PTPs should be validated against subs...,Observed PTP fulfillment rate = 7.09%.,PTP volume alone may overstate actual collecti...,Track PTP fulfillment using independently matc...
5,F-06,Data Quality,Data-quality and attribution limitations affec...,"Payment, timestamp, agent identity, and attrib...",Unresolved data issues can distort recovery an...,Maintain explicit data-quality flags and metri...
6,F-07,Payments,Repeated payment references are not confirmed ...,"3,284 repeated references; 0 exact duplicate r...",Automatic reference-based deduplication could ...,Do not deduplicate payment references without ...
7,F-08,Timestamps,Payment timestamps are timezone-naive,"0 timezone-labelled records; 5,213 events (21....",Calendar-date analysis may be sensitive to an ...,Preserve source timestamps and document the ti...
8,F-09,Attribution,Multiple attribution records can double-count ...,25 payments with multiple attribution records;...,Raw attributed recovery can overstate campaign...,Use successful payments as the source of truth...
